# Agente de Triagem de Manutenção — Entrega 2

## Orquestração com LangGraph e Sistema Multiagente

**Engenharia de Agentes e IA Agêntica** · Pós-graduação · PUC Minas · Prof. Paulo Moreira

| | |
|---|---|
| **Empresa (fictícia)** | Construtora Apollo S.A. |
| **Ambiente** | Google Colab |
| **Modelo** | `gemini-3.6-flash` (reservas: `gemini-3.5-flash` e `gemini-3.1-flash-lite`) · embeddings `gemini-embedding-001` |
| **Orquestração** | LangChain + LangGraph |
| **Integração** | MCP (agente ↔ ferramentas) · A2A (agente ↔ agente) |
| **Repositório** | [github.com/diogo-hsf/apollo-agente-manutencao](https://github.com/diogo-hsf/apollo-agente-manutencao) |
| **Requisito externo** | Apenas a `GEMINI_API_KEY` |

> **Como executar.** Cadastre a chave do Gemini nos *Secrets* do Colab com o nome `GEMINI_API_KEY`, conceda acesso ao notebook quando solicitado e use *Ambiente de execução → Executar tudo*.

Esta entrega evolui o agente da Entrega 1 em duas partes, na ordem pedida no enunciado:

| | O que muda |
|---|---|
| **Entrega 1** | Ponto de partida: um agente, com o laço escrito em Python |
| **Parte 1** | O mesmo agente, agora orquestrado por LangGraph |
| **Parte 2** | A triagem dividida entre um Supervisor e agentes especialistas |

## Sumário

1. Por que evoluir
2. Desenho da Parte 1 — LangGraph
3. Desenho da Parte 2 — Sistema multiagente
4. Configuração do ambiente
5. Dados operacionais
6. Base de conhecimento técnica
7. Servidor e cliente MCP
8. Parte 1 — O agente em LangGraph
9. Parte 2 — Os agentes especialistas

---
# 1. Por que evoluir

## 1.1 O ponto de partida

Na Entrega 1, o agente já consultava a documentação (Vector RAG), usava ferramentas e validava o próprio parecer. O que ficou difícil de manter foi a orquestração: estado, roteamento e o ciclo modelo → ferramenta → modelo ficaram espalhados em métodos Python, e cada regra nova de fluxo virava mais um `if` dentro do laço.

Além disso, um único agente concentra três responsabilidades de natureza diferente:

| Responsabilidade | Pergunta | Fonte |
|---|---|---|
| Conhecimento técnico | O que dizem as fichas, o catálogo de alarmes, o procedimento de segurança e a política? | Documentos (RAG) |
| Fatos operacionais | O que dizem o cadastro, o histórico, a preventiva, o estoque e a frota? | Sistemas (MCP) |
| Parecer | Dadas as regras e os fatos, o que recomendar ao planejador? | As evidências recebidas |

## 1.2 Por que separar em agentes

Multiagente é uma decisão de arquitetura, não um nível obrigatório de maturidade. Neste caso, a separação se justifica pelos critérios abaixo:

| Critério | Na triagem da Apollo |
|---|---|
| Conhecimento especializado | Documentação técnica e dados operacionais são domínios diferentes, com fontes diferentes |
| Ferramentas e permissões | Só o agente de operações acessa os sistemas; só o de conhecimento acessa os documentos; o agente do parecer não acessa nenhum dos dois. Isso reduz o risco visto na Entrega 1 de afirmar um fato que nenhuma ferramenta confirmou |
| Isolamento de contexto | Cada especialista recebe só o contexto da sua subtarefa |
| Execução paralela | Conhecimento e operações não dependem um do outro e rodam ao mesmo tempo |
| Governança | Permissões por agente e registro das decisões de roteamento, base para a entrega final |

O custo é maior: mais chamadas ao modelo por OS, mais latência e um problema novo de coordenação. Também muda quem decide: na Entrega 1, o agente escolhia as ferramentas a cada rodada; na Parte 2, essa decisão passa para o Supervisor, que escolhe quais especialistas acionar e quando encerrar. O agente de operações executa um conjunto fixo de consultas.

## 1.3 O que continua como etapa, e não como agente

Nem toda etapa precisa virar agente. Continuam como etapas determinísticas:

| Etapa | Motivo |
|---|---|
| Percepção da OS | Regra fixa: valida a OS, extrai alarmes, resolve o equipamento e detecta gatilhos de segurança |
| Validação do parecer | Regra de verificação de schema e evidências |
| Guardrail de segurança | Trava fixa, que não pode depender do julgamento do modelo |
| Gravação na fila | Ação de persistência |
| Cada consulta aos sistemas | É uma ferramenta, servida por MCP |

---
# 2. Desenho da Parte 1 — LangGraph

Na Parte 1, a capacidade de negócio fica como está e só a orquestração muda.

```text
START → PERCEIVE ─┬─ OS válida → MODEL ─┬─ pediu ferramentas → TOOLS → MODEL
                  │                      └─ não pediu, ou limite → FINALIZE → VALIDATE → ACTION → END
                  └─ OS inválida ─────────────────────────────────────────────────────→ ACTION → END
```

| Elemento | Entrega 1 | Parte 1 |
|---|---|---|
| Ferramentas | Registro `nome → função` | Servidor MCP com as 6 capacidades, cliente MCP e adaptador para `StructuredTool` |
| Estado | `EstadoTriagem` | `TriagemState` (`TypedDict`), com `messages` e o reducer `add_messages` |
| Modelo | SDK `google-genai` | `ChatGoogleGenerativeAI` com `bind_tools()` |
| Ciclo | `while` no método `_decidir()` | Edge `tools → model` |
| Roteamento | `if` dentro do laço | Conditional edges depois de `perceive` e de `model` |
| Finalização | Ferramenta terminal `emitir_parecer` | Node `finalize` com `with_structured_output()` |
| Validação e guardrail | Funções no fim do método | Node `validate` |
| Fila | `agir()` | Node `action` |

A resiliência da Entrega 1 continua: novas tentativas em erros 503 e 429, modelos de reserva e parecer de contingência quando a triagem não pode ser concluída.

---
# 3. Desenho da Parte 2 — Sistema multiagente

## 3.1 Topologia

Entre as topologias possíveis (supervisor, peer-to-peer e blackboard), a triagem pede **supervisor**: o fluxo precisa ser controlado, cada subtarefa tem um responsável conhecido e o resultado é um parecer único. Não há negociação, porque cada responsabilidade já tem um especialista definido.

```text
                         OS de manutenção
                                │
                ┌───────────────▼────────────────┐
                │        SUPERVISOR AGENT        │   LangGraph interno
                │  planejador Gemini + A2A Client │
                └──────┬───────────┬──────────┬──┘
                      A2A         A2A        A2A
              ┌────────▼───┐ ┌─────▼────┐ ┌───▼───────┐
              │ KNOWLEDGE  │ │OPERATIONS│ │ STRATEGY  │   A2A Servers
              │ Vector RAG │ │MCP Client│ │ sem tools │
              └────────────┘ └────┬─────┘ └───────────┘
                                  │ MCP
                    cadastro · histórico · preventiva · estoque · frota
```

## 3.2 Agentes

| Agente | Objetivo | Acessa | Não acessa |
|---|---|---|---|
| **Supervisor** | Decidir quais especialistas acionar, delegar e consolidar | Agent Cards e Artifacts recebidos | RAG, MCP, sistemas |
| **Knowledge** | Interpretar a documentação técnica aplicável à OS | Vector RAG + Gemini | Sistemas operacionais |
| **Operations** | Levantar os fatos operacionais do equipamento | Cliente MCP com as capacidades operacionais | Documentação |
| **Strategy** | Emitir o parecer a partir das evidências | Só os Artifacts repassados pelo Supervisor | RAG, MCP |

## 3.3 Contratos de interação

A comunicação segue o modelo A2A: o Supervisor envia uma *Message* com um payload JSON, o especialista abre uma *Task* e devolve um *Artifact* JSON. Um `context_id` liga todas as tasks da mesma OS.

| Agente | Entrada | Saída |
|---|---|---|
| **Knowledge** | `objective` e a percepção da OS (relato, alarmes, gatilhos, ficha técnica do equipamento) | Resumo técnico, orientações, peças associadas e referências recuperadas |
| **Operations** | `objective`, equipamento informado e obra | Cadastro (ou candidatos), histórico, preventiva, estoque das peças da ficha técnica e frota |
| **Strategy** | `objective`, percepção e os Artifacts de Knowledge e Operations | Parecer no mesmo schema da Entrega 1 |

Cada agente recebe só o necessário: o Knowledge não recebe fatos operacionais, o Operations recebe só os identificadores e apenas o Strategy recebe os dois Artifacts. As referências citadas no parecer são conferidas contra as recuperadas pelo Knowledge, e os gatilhos de segurança da percepção seguem até o guardrail.

Duas adaptações foram necessárias para o caso da Apollo:

- **Estoque por ficha técnica.** Com consultas fixas, o Operations não sabe de antemão quais peças consultar. A capacidade de estoque passou a aceitar a ficha técnica e devolve as peças associadas a ela.
- **Três buscas no Knowledge.** Na Entrega 1, uma pergunta genérica não encontrou a seção certa da documentação. O Knowledge faz três buscas fixas (sintoma ou alarme na ficha técnica; prioridade e recorrência; critérios de bloqueio) e uma única síntese pelo Gemini.

## 3.4 Coordenação

| Mecanismo | Na triagem |
|---|---|
| Delegação | O Supervisor envia objetivo e contexto mínimo, sem dizer como executar |
| Paralelismo | Knowledge e Operations na mesma rodada |
| Sincronização | O Strategy só depois dos dois Artifacts |
| Validação do plano | Determinística: agente existe, não foi repetido, objetivo não vazio |
| Limite | Número máximo de ciclos do Supervisor |
| Registro | *Routing log* com as decisões do Supervisor |

---
# 4. Configuração do ambiente

## 4.1 Dependências

Versões fixadas para que o notebook se comporte igual em qualquer execução.

In [ ]:
%pip install -q \
  "google-genai==2.25.0" \
  "langchain==1.4.2" \
  "langgraph==1.2.12" \
  "langchain-google-genai==4.4.0" \
  "mcp==2.2.0" \
  "a2a-sdk[http-server]==1.2.0" \
  "chromadb>=1.0,<2" "pypdf>=5,<7" "jsonschema>=4.23,<5" \
  pandas uvicorn starlette httpx grandalf

## 4.2 Importações

In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import json
import logging
import os
import random
import re
import subprocess
import sys
import unicodedata
import urllib.request
import warnings
from dataclasses import asdict, dataclass
from datetime import date
from pathlib import Path
from typing import Annotated, Any, Callable, TypedDict

import chromadb
import httpx
import pandas as pd
from google import genai
from google.genai import errors, types
from IPython.display import Image, display
from jsonschema import Draft202012Validator
from pypdf import PdfReader

# LangChain
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langchain_google_genai import ChatGoogleGenerativeAI

# LangGraph
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.prebuilt import ToolNode

# A2A Client
from a2a.client import A2ACardResolver, ClientCallContext, ClientConfig, create_client
from a2a.helpers import get_artifact_text, get_message_text
from a2a.types import Message, Part, Role, SendMessageRequest, TaskState

# Avisos internos do google-genai que não se aplicam aqui: o laço de ferramentas é controlado pelo
# grafo, e a resposta de erro assíncrona do SDK não afeta a nova tentativa.
logging.getLogger("google_genai.models").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="coroutine 'ClientResponse.json' was never awaited")

print("Bibliotecas importadas.")

## 4.3 Parâmetros

`DATA_REFERENCIA` é o "hoje" do caso, como na Entrega 1. `MODELOS` lista os modelos de geração em ordem de preferência: o primeiro é o principal e os demais só entram se ele ficar indisponível.

In [ ]:
REPOSITORIO_RAW = "https://raw.githubusercontent.com/diogo-hsf/apollo-agente-manutencao/main/"
PASTA_LOCAL = Path("apollo_dados")
DATA_REFERENCIA = date(2026, 9, 28)

MODELOS = ["gemini-3.6-flash", "gemini-3.5-flash", "gemini-3.1-flash-lite"]
MODELO = MODELOS[0]
MODELO_EMBEDDING = "gemini-embedding-001"
DIMENSAO_EMBEDDING = 768

# Resiliência (Entrega 1)
TENTATIVAS_POR_CHAMADA = 3
ESPERA_BASE_SEGUNDOS = 30.0          # esperas de 30 s e 60 s
ESPERA_MAXIMA_ACEITAVEL = 90         # retryDelay maior que isso = cota diária
TIMEOUT_CHAMADA_S = 180

DIRETORIO_CHROMA = "chroma_db"
NOME_COLECAO = "apollo_base_tecnica"

print(f"Modelo principal: {MODELO} · reservas: {', '.join(MODELOS[1:])}")

## 4.4 Chave da API e conexão

Mesmo procedimento da Entrega 1: a chave vem dos *Secrets* do Colab e a disponibilidade de cada modelo é verificada com `models.get`, sem gastar cota.

In [ ]:
def obter_api_key() -> str:
    """Recupera a chave do Gemini sem expô-la no notebook."""
    try:
        from google.colab import userdata  # type: ignore
        chave = userdata.get("GEMINI_API_KEY")
        if chave:
            print("Chave obtida dos Secrets do Colab.")
            return chave
    except Exception:
        pass
    chave = os.environ.get("GEMINI_API_KEY")
    if chave:
        print("Chave obtida da variável de ambiente GEMINI_API_KEY.")
        return chave
    from getpass import getpass
    chave = getpass("Cole sua GEMINI_API_KEY (a digitação fica oculta): ").strip()
    if not chave:
        raise RuntimeError("Nenhuma GEMINI_API_KEY foi fornecida.")
    return chave


GEMINI_API_KEY = obter_api_key()
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

cliente_genai = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(timeout=TIMEOUT_CHAMADA_S * 1000,
                                   retry_options=types.HttpRetryOptions(attempts=1)),
)

MODELOS_DISPONIVEIS: list[str] = []
for modelo in MODELOS:
    try:
        cliente_genai.models.get(model=modelo)
        MODELOS_DISPONIVEIS.append(modelo)
        print(f"  {modelo:<24} disponível")
    except errors.APIError as erro:
        print(f"  {modelo:<24} indisponível para esta chave (HTTP {erro.code})")
if not MODELOS_DISPONIVEIS:
    raise RuntimeError(f"Nenhum dos modelos {MODELOS} está acessível. Confira a GEMINI_API_KEY.")

## 4.5 Acesso ao modelo com resiliência

A Entrega 1 tinha uma camada própria para chamar o Gemini, com novas tentativas, modelos de reserva e detecção de cota esgotada. Aqui ela foi adaptada para o LangChain: o `ClienteLLM` cria um `ChatGoogleGenerativeAI` por modelo e aplica as mesmas regras sobre qualquer chamada.

| Situação | Resposta |
|---|---|
| 503, 500, 502, 504 ou 429 por minuto | Nova tentativa após 30 s e 60 s |
| Timeout ou queda de conexão | Nova tentativa, como um 503 |
| Cota diária esgotada (429 com espera de horas) | Sem nova tentativa; passa para o modelo de reserva |
| Modelo indisponível após as tentativas | Passa para o modelo de reserva |
| Erro do cliente (400, 403) | Propaga na hora |

O `ChatGoogleGenerativeAI` é criado com `max_retries=1`, que desliga o retry interno do SDK. Sem isso, cada erro seria repetido duas vezes e a cota acabaria mais rápido.

A troca de modelo só acontece no início de uma conversa. No meio dela, o histórico já tem metadados de raciocínio do modelo anterior, e a triagem termina em contingência.

In [ ]:
class CotaEsgotadaError(RuntimeError):
    """Cota do modelo esgotada, sem liberação em tempo útil."""


CODIGOS_TRANSITORIOS = {429, 500, 502, 503, 504}


def _codigo_http(erro: BaseException) -> int | None:
    """Código HTTP do erro, mesmo quando o LangChain o envolve em outra exceção."""
    for e in (erro, erro.__cause__):
        codigo = getattr(e, "code", None)
        if isinstance(codigo, int):
            return codigo
    return None


def _retry_delay_segundos(erro: BaseException) -> float | None:
    texto = " ".join(str(e) + json.dumps(getattr(e, "details", None) or {}, default=str)
                     for e in (erro, erro.__cause__) if e is not None)
    achado = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", texto)
    return float(achado.group(1)) if achado else None


def _eh_transitorio(erro: BaseException) -> bool:
    return (isinstance(erro, (httpx.TimeoutException, httpx.TransportError))
            or _codigo_http(erro) in CODIGOS_TRANSITORIOS)


async def chamar_com_retry(funcao, *, descricao: str, tentativas: int = TENTATIVAS_POR_CHAMADA,
                           espera_base: float = ESPERA_BASE_SEGUNDOS, dormir=asyncio.sleep):
    """Executa uma chamada assíncrona com novas tentativas para erros transitórios."""
    for tentativa in range(1, tentativas + 1):
        try:
            return await funcao()
        except Exception as erro:
            if not _eh_transitorio(erro):
                raise
            sugerido = _retry_delay_segundos(erro)
            if _codigo_http(erro) == 429 and sugerido and sugerido > ESPERA_MAXIMA_ACEITAVEL:
                raise CotaEsgotadaError(
                    f"Cota esgotada em '{descricao}': a API pede {sugerido / 3600:.1f} h de espera.") from erro
            if tentativa == tentativas:
                raise
            espera = (sugerido or espera_base * 2 ** (tentativa - 1)) + random.uniform(0, 1)
            motivo = _codigo_http(erro) or type(erro).__name__
            print(f"  [{descricao}] {motivo} — tentativa {tentativa}/{tentativas}; nova tentativa em {espera:.0f}s")
            await dormir(espera)
    raise RuntimeError(f"[{descricao}] falha após {tentativas} tentativas.")


class ClienteLLM:
    """Acesso ao Gemini: um ChatGoogleGenerativeAI por modelo, com retry, reserva e contagem."""

    def __init__(self, modelos: list[str], dormir=asyncio.sleep):
        self.modelos = list(modelos)
        self._indice = 0
        self._chats: dict[str, ChatGoogleGenerativeAI] = {}
        self._dormir = dormir
        self.chamadas_geracao = 0
        self.chamadas_embedding = 0
        self.trocas_de_modelo: list[str] = []
        self.cota_esgotada = False

    @property
    def modelo(self) -> str:
        return self.modelos[self._indice]

    def chat(self, modelo: str) -> ChatGoogleGenerativeAI:
        if modelo not in self._chats:
            self._chats[modelo] = ChatGoogleGenerativeAI(
                model=modelo, api_key=GEMINI_API_KEY, max_retries=1, timeout=TIMEOUT_CHAMADA_S)
        return self._chats[modelo]

    def _passar_para_reserva(self, motivo: str) -> bool:
        if self._indice + 1 >= len(self.modelos):
            return False
        anterior = self.modelo
        self._indice += 1
        self.trocas_de_modelo.append(f"{anterior} → {self.modelo} ({motivo})")
        print(f"  [geração] {anterior} indisponível ({motivo}); as próximas chamadas usam {self.modelo}.")
        return True

    async def invocar(self, montar: Callable, mensagens: list[BaseMessage], *, inicio_da_conversa: bool):
        """Chama o modelo atual. `montar` recebe o chat e devolve o runnable (bind_tools, structured output)."""
        if self.cota_esgotada:
            raise CotaEsgotadaError("a cota diária dos modelos já se esgotou nesta sessão; chamada não enviada.")
        while True:
            modelo = self.modelo
            runnable = montar(self.chat(modelo))
            self.chamadas_geracao += 1
            try:
                return await chamar_com_retry(lambda: runnable.ainvoke(mensagens),
                                              descricao=f"geração · {modelo}", dormir=self._dormir)
            except CotaEsgotadaError:
                if not self._passar_para_reserva("cota diária esgotada"):
                    self.cota_esgotada = True
                    raise
                if not inicio_da_conversa:
                    raise
            except Exception as erro:
                if not _eh_transitorio(erro):
                    raise
                if not self._passar_para_reserva(f"HTTP {_codigo_http(erro)}" if _codigo_http(erro) else type(erro).__name__) \
                        or not inicio_da_conversa:
                    raise

    async def embed(self, textos: list[str], tipo: str, lote: int = 100) -> list[list[float]]:
        """Embeddings em lotes ('RETRIEVAL_DOCUMENT' ou 'RETRIEVAL_QUERY')."""
        vetores: list[list[float]] = []
        for inicio in range(0, len(textos), lote):
            parte = textos[inicio:inicio + lote]
            resposta = await chamar_com_retry(
                lambda: cliente_genai.aio.models.embed_content(
                    model=MODELO_EMBEDDING, contents=parte,
                    config=types.EmbedContentConfig(task_type=tipo, output_dimensionality=DIMENSAO_EMBEDDING)),
                descricao="embedding", dormir=self._dormir)
            self.chamadas_embedding += 1
            vetores.extend(list(e.values) for e in resposta.embeddings)
        return vetores

    def resumo(self) -> str:
        texto = (f"modelo: {self.modelo} · chamadas de geração: {self.chamadas_geracao} · "
                 f"requisições de embedding: {self.chamadas_embedding}")
        if self.trocas_de_modelo:
            texto += f" · trocas de modelo: {len(self.trocas_de_modelo)}"
        if self.cota_esgotada:
            texto += " · cota diária esgotada"
        return texto


llm = ClienteLLM(MODELOS_DISPONIVEIS)
print("ClienteLLM pronto —", llm.resumo())

---
# 5. Dados operacionais

Mesmas tabelas da Entrega 1, baixadas do repositório e validadas contra o contrato de cada uma antes do uso. Os arquivos ficam em `apollo_dados/`, onde o servidor MCP também vai lê-los.

In [ ]:
def baixar_arquivo(caminho_relativo: str, tentativas: int = 3) -> Path:
    """Baixa um arquivo do repositório; em falha, usa a cópia local, se houver."""
    destino = PASTA_LOCAL / caminho_relativo
    destino.parent.mkdir(parents=True, exist_ok=True)
    url = REPOSITORIO_RAW + caminho_relativo
    ultimo_erro: Exception | None = None
    for tentativa in range(1, tentativas + 1):
        try:
            with urllib.request.urlopen(url, timeout=30) as resposta:
                conteudo = resposta.read()
            if not conteudo:
                raise ValueError("arquivo vazio")
            destino.write_bytes(conteudo)
            return destino
        except Exception as erro:
            ultimo_erro = erro
    if destino.exists() and destino.stat().st_size > 0:
        print(f"  aviso: download de {caminho_relativo} falhou ({ultimo_erro}); usando cópia local.")
        return destino
    raise RuntimeError(f"Não foi possível obter {caminho_relativo} ({url}): {ultimo_erro}")


class DadosInvalidosError(ValueError):
    """Tabela fora do contrato esperado."""


CONTRATO_TABELAS: dict[str, dict[str, list[str]]] = {
    "obras": {"colunas": ["obra_id", "nome", "municipio_uf", "almoxarifado_id"],
              "numericas": [], "chave": ["obra_id"]},
    "equipamentos": {"colunas": ["equipamento_id", "tipo", "fabricante", "modelo", "ano_fabricacao", "obra_id",
                                 "status_operacional", "criticidade", "horimetro_atual", "ficha_tecnica"],
                     "numericas": ["ano_fabricacao", "horimetro_atual"], "chave": ["equipamento_id"]},
    "ordens_servico": {"colunas": ["os_id", "data_abertura", "obra_id", "equipamento_informado", "aberta_por",
                                   "relato"], "numericas": [], "chave": ["os_id"]},
    "historico_manutencao": {"colunas": ["ordem_id", "equipamento_id", "data_abertura", "data_conclusao",
                                         "tipo_manutencao", "sistema", "codigo_alarme", "descricao_servico",
                                         "pecas_utilizadas", "horas_parado"], "numericas": [], "chave": ["ordem_id"]},
    "plano_preventiva": {"colunas": ["ficha_tecnica", "revisao", "intervalo_horas"],
                         "numericas": ["intervalo_horas"], "chave": ["ficha_tecnica", "revisao"]},
    "registro_preventivas": {"colunas": ["equipamento_id", "revisao", "horimetro_execucao", "data_execucao"],
                             "numericas": ["horimetro_execucao"], "chave": ["equipamento_id", "revisao"]},
    "estoque_pecas": {"colunas": ["codigo_peca", "descricao", "ref_fabricante", "ficha_tecnica", "almoxarifado_id",
                                  "saldo", "unidade", "prazo_reposicao_dias"],
                      "numericas": ["saldo", "prazo_reposicao_dias"], "chave": ["codigo_peca", "almoxarifado_id"]},
}


def carregar_tabela(nome: str) -> pd.DataFrame:
    contrato = CONTRATO_TABELAS[nome]
    df = pd.read_csv(baixar_arquivo(f"dados/{nome}.csv"), dtype=str, keep_default_na=False, encoding="utf-8")
    faltando = [c for c in contrato["colunas"] if c not in df.columns]
    if faltando:
        raise DadosInvalidosError(f"{nome}: colunas ausentes {faltando}")
    for coluna in contrato["numericas"]:
        convertida = pd.to_numeric(df[coluna], errors="coerce")
        if convertida.isna().any():
            raise DadosInvalidosError(f"{nome}.{coluna}: valores não numéricos")
        df[coluna] = convertida.astype(int)
    if df.duplicated(subset=contrato["chave"]).any():
        raise DadosInvalidosError(f"{nome}: chave {contrato['chave']} duplicada")
    return df


TABELAS = {nome: carregar_tabela(nome) for nome in CONTRATO_TABELAS}
PASTA_DADOS = (PASTA_LOCAL / "dados").resolve()

print("Tabelas carregadas e validadas:")
for nome, df in TABELAS.items():
    print(f"  {nome:<22} {len(df):>3} linhas")

In [ ]:
with pd.option_context("display.max_colwidth", 110):
    display(TABELAS["ordens_servico"][["os_id", "obra_id", "equipamento_informado", "relato"]])

---
# 6. Base de conhecimento técnica

Mesma base da Entrega 1: os sete documentos técnicos, divididos por seção e indexados no ChromaDB. A indexação acontece aqui, no notebook. As consultas são feitas depois pelo servidor MCP (Parte 1) e pelo Knowledge Agent (Parte 2), que leem o mesmo índice.

In [ ]:
DOCUMENTOS_BASE: dict[str, str] = {
    "FTM-CAT320": "Ficha Técnica de Manutenção — Escavadeira Hidráulica CAT 320",
    "FTM-CATD9T": "Ficha Técnica de Manutenção — Trator de Esteiras CAT D9T",
    "FTM-CAT924H": "Ficha Técnica de Manutenção — Carregadeira de Rodas CAT 924H",
    "FTM-MBCAM": "Ficha Técnica de Manutenção — Caminhões Basculantes Mercedes-Benz",
    "CAT-ALM-001": "Catálogo de Alarmes de Telemetria e Ocorrências",
    "PRO-SEG-001": "Procedimento de Segurança — Bloqueio e Impedimento de Operação",
    "POL-MAN-001": "Política de Manutenção da Frota de Equipamentos",
}

RE_SECAO = re.compile(r"^(\d{1,2}) ([A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-ZÁÉÍÓÚÂÊÔÃÕÇ ,—\-]+)$")
RE_SUBSECAO = re.compile(r"^(\d{1,2}\.\d{1,2}) ([A-ZÁÉÍÓÚÂÊÔÃÕÇ].*)$")
RE_RUIDO = re.compile(r"^(Construtora Apollo S\.A\. · .*|Página \d+)$")


def segmentar_documento(codigo: str, caminho: Path) -> list[dict[str, Any]]:
    """Divide o PDF em chunks por seção, com metadados para filtro e citação (Entrega 1)."""
    secoes: list[dict[str, Any]] = []
    titulo_pai = ""
    for pagina in PdfReader(caminho).pages:
        for linha in (pagina.extract_text() or "").splitlines():
            linha = linha.strip()
            if not linha or RE_RUIDO.match(linha):
                continue
            m_secao, m_sub = RE_SECAO.match(linha), RE_SUBSECAO.match(linha)
            if m_secao:
                titulo_pai = m_secao.group(2).capitalize()
                secoes.append({"secao": m_secao.group(1), "titulo": titulo_pai, "linhas": []})
            elif m_sub:
                secoes.append({"secao": m_sub.group(1), "titulo": f"{titulo_pai} › {m_sub.group(2)}", "linhas": []})
            elif secoes:
                secoes[-1]["linhas"].append(linha)
    aplicacao = codigo if codigo.startswith("FTM-") else "geral"
    chunks = []
    for secao in secoes:
        texto = "\n".join(secao["linhas"]).strip()
        if not texto:
            continue
        assinatura = hashlib.sha1(texto.encode("utf-8")).hexdigest()[:8]
        chunks.append({
            "id": f"{codigo}§{secao['secao']}#1-{assinatura}",
            "texto": texto,
            "metadata": {"documento": codigo, "titulo_documento": DOCUMENTOS_BASE[codigo],
                         "secao": secao["secao"], "titulo": secao["titulo"],
                         "referencia": f"{codigo} § {secao['secao']}", "aplicacao": aplicacao},
        })
    return chunks


ARQUIVOS_PDF = {codigo: baixar_arquivo(f"documentos/{codigo}.pdf") for codigo in DOCUMENTOS_BASE}
CHUNKS = [c for codigo, caminho in ARQUIVOS_PDF.items() for c in segmentar_documento(codigo, caminho)]

colecao = chromadb.PersistentClient(path=DIRETORIO_CHROMA).get_or_create_collection(
    name=NOME_COLECAO, metadata={"hnsw:space": "cosine"})
ids_atuais = [c["id"] for c in CHUNKS]
obsoletos = sorted(set(colecao.get()["ids"]) - set(ids_atuais))
if obsoletos:
    colecao.delete(ids=obsoletos)
ja_indexados = set(colecao.get(ids=ids_atuais)["ids"])
novos = [c for c in CHUNKS if c["id"] not in ja_indexados]
if novos:
    vetores = await llm.embed(
        [f"{c['metadata']['titulo_documento']} — {c['metadata']['titulo']}\n{c['texto']}" for c in novos],
        tipo="RETRIEVAL_DOCUMENT")
    colecao.upsert(ids=[c["id"] for c in novos], embeddings=vetores,
                   documents=[c["texto"] for c in novos], metadatas=[c["metadata"] for c in novos])

print(f"{len(ARQUIVOS_PDF)} documentos · {colecao.count()} chunks no índice · novos nesta execução: {len(novos)}")
print(llm.resumo())

---
# 7. Servidor e cliente MCP

## 7.1 Servidor MCP

As seis capacidades da Entrega 1 passam a ser servidas por um servidor MCP, executado como subprocesso stdio. O servidor é a fonte de verdade dos contratos: nome, descrição e schema de entrada de cada ferramenta.

O que veio da Entrega 1:

- a mesma lógica de cada ferramenta e o status padronizado (`success`, `not_found`, `invalid_arguments`, `error`);
- o isolamento de exceções e a conferência do retorno antes de devolvê-lo;
- os limites dos argumentos (tamanho da pergunta, formato `APL-00000`, janela de 1 a 365 dias), agora declarados nos tipos dos parâmetros e publicados no schema.

A única mudança de contrato é a aprovada no desenho: `consultar_estoque_pecas` também aceita a ficha técnica e devolve as peças associadas a ela.

O servidor tem dois perfis. No perfil `completo`, usado pelo agente único da Parte 1, publica as seis capacidades. No perfil `operacional`, usado pelo Operations Agent da Parte 2, publica só as cinco capacidades operacionais: assim o Operations não tem acesso à documentação, como definido no desenho.

In [ ]:
SERVIDOR_MCP = Path("servidor_mcp_apollo.py")

CODIGO_SERVIDOR_MCP = r'''"""Servidor MCP da Construtora Apollo — capacidades da triagem de manutenção.

Executado como subprocesso stdio. É a fonte de verdade dos contratos das ferramentas
(nome, descrição e schema de entrada), descobertos pelo cliente em tempo de execução.

Perfis (variável APOLLO_MCP_PERFIL):
- completo: as seis capacidades, usadas pelo agente único da Parte 1;
- operacional: só as cinco capacidades operacionais, usadas pelo Operations Agent da Parte 2.
"""
from __future__ import annotations

import json
import os
import re
import sys
import time
from functools import lru_cache
from typing import Annotated, Any, Callable, Literal

import chromadb
import pandas as pd
from google import genai
from google.genai import types
from mcp.server import MCPServer
from pydantic import Field

PERFIL = os.environ.get("APOLLO_MCP_PERFIL", "completo")
PASTA_DADOS = os.environ["APOLLO_DADOS"]
DATA_REFERENCIA = pd.Timestamp(os.environ["APOLLO_DATA_REFERENCIA"])

MODELO_EMBEDDING = "gemini-embedding-001"
DIMENSAO_EMBEDDING = 768
JANELA_RECORRENCIA_DIAS = 90          # POL-MAN-001 § 6
TOLERANCIA_PREVENTIVA = 0.10          # POL-MAN-001 § 7
LIMITE_TRECHO = 1500
LIMITE_RETORNO = 8000

mcp = MCPServer("apollo-manutencao" if PERFIL == "completo" else "apollo-manutencao-operacional")

if PERFIL == "completo":           # só o perfil completo acessa a base de conhecimento
    cliente_gemini = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    colecao = chromadb.PersistentClient(path=os.environ["APOLLO_CHROMA_DIR"]).get_collection(
        os.environ["APOLLO_CHROMA_COLECAO"])


def _ler(nome: str, numericas: tuple[str, ...] = ()) -> pd.DataFrame:
    df = pd.read_csv(os.path.join(PASTA_DADOS, f"{nome}.csv"), dtype=str,
                     keep_default_na=False, encoding="utf-8")
    for coluna in numericas:
        df[coluna] = pd.to_numeric(df[coluna]).astype(int)
    return df


TABELAS = {
    "obras": _ler("obras"),
    "equipamentos": _ler("equipamentos", ("ano_fabricacao", "horimetro_atual")),
    "historico_manutencao": _ler("historico_manutencao"),
    "plano_preventiva": _ler("plano_preventiva", ("intervalo_horas",)),
    "registro_preventivas": _ler("registro_preventivas", ("horimetro_execucao",)),
    "estoque_pecas": _ler("estoque_pecas", ("saldo", "prazo_reposicao_dias")),
}

SISTEMAS = ("Hidráulico", "Motor", "Arrefecimento", "Elétrico", "Trem de força", "Freios",
            "Direção", "Suspensão", "Material rodante", "Estrutura e implementos")
OBRAS = tuple(sorted(TABELAS["obras"]["obra_id"]))
TIPOS = tuple(sorted(TABELAS["equipamentos"]["tipo"].unique()))
FICHAS = tuple(sorted(TABELAS["equipamentos"]["ficha_tecnica"].unique()))


# ---------------------------------------------------------------------------
# Helpers (Entrega 1)
# ---------------------------------------------------------------------------

def _sucesso(**dados) -> dict[str, Any]:
    return {"status": "success", **dados}


def _nao_encontrado(mensagem: str, **dados) -> dict[str, Any]:
    return {"status": "not_found", "mensagem": mensagem, **dados}


def _argumento_invalido(mensagem: str) -> dict[str, Any]:
    return {"status": "invalid_arguments", "mensagem": mensagem}


def _normalizar_id_equipamento(valor: str) -> str:
    texto = re.sub(r"[\s_]+", "-", valor.strip().upper())
    if re.fullmatch(r"[A-Z]{2}\d{2}", texto):
        texto = f"{texto[:2]}-{texto[2:]}"
    return texto


def _encurtar(texto: str, limite: int = LIMITE_TRECHO) -> str:
    if len(texto) <= limite:
        return texto
    return texto[:limite].rsplit(" ", 1)[0] + " […]"


def _localizar_equipamento(equipamento_id: str) -> pd.Series | None:
    eq = TABELAS["equipamentos"]
    linha = eq[eq["equipamento_id"] == _normalizar_id_equipamento(equipamento_id)]
    return None if linha.empty else linha.iloc[0]


def _resumo_equipamento(linha: pd.Series) -> dict[str, Any]:
    return {
        "equipamento_id": linha["equipamento_id"], "tipo": linha["tipo"], "modelo": linha["modelo"],
        "obra_id": linha["obra_id"], "status_operacional": linha["status_operacional"],
        "criticidade": linha["criticidade"], "horimetro_atual": int(linha["horimetro_atual"]),
        "ficha_tecnica": linha["ficha_tecnica"],
    }


def _executar(nome: str, campos_sucesso: tuple[str, ...], funcao: Callable[[], dict[str, Any]]) -> dict[str, Any]:
    """Isola exceções e confere o retorno contra o contrato antes de devolvê-lo (Entrega 1, 6.4)."""
    try:
        resultado = funcao()
    except Exception as erro:
        print(f"[{nome}] erro interno: {type(erro).__name__}: {erro}", file=sys.stderr)
        return {"status": "error", "mensagem": "Falha interna ao executar a ferramenta. Prossiga com as "
                                               "demais fontes e registre esta verificação como pendência."}
    problema = None
    if not isinstance(resultado, dict) or resultado.get("status") not in (
            "success", "not_found", "invalid_arguments", "error"):
        problema = "status fora do contrato"
    elif resultado["status"] == "success" and any(c not in resultado for c in campos_sucesso):
        problema = "campos obrigatórios ausentes"
    elif len(json.dumps(resultado, ensure_ascii=False, default=str)) > LIMITE_RETORNO:
        problema = "retorno acima do tamanho máximo"
    if problema:
        print(f"[{nome}] retorno fora do contrato: {problema}", file=sys.stderr)
        return {"status": "error", "mensagem": "A ferramenta devolveu um resultado fora do contrato, que foi "
                                               "descartado. Registre esta verificação como pendência."}
    return resultado


@lru_cache(maxsize=256)
def _embedding_consulta(pergunta: str) -> tuple[float, ...]:
    for tentativa in range(1, 4):
        try:
            resposta = cliente_gemini.models.embed_content(
                model=MODELO_EMBEDDING, contents=[pergunta],
                config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY",
                                                output_dimensionality=DIMENSAO_EMBEDDING))
            return tuple(resposta.embeddings[0].values)
        except Exception:
            if tentativa == 3:
                raise
            time.sleep(10 * tentativa)
    raise RuntimeError("falha no embedding")


# ---------------------------------------------------------------------------
# Capacidades
# ---------------------------------------------------------------------------

def buscar_documentacao_tecnica(
    pergunta: Annotated[str, Field(min_length=10, max_length=300,
                                   description="Pergunta específica sobre um único assunto.")],
    ficha_tecnica: Annotated[Literal[FICHAS] | None, Field(
        description="Ficha técnica do equipamento (campo ficha_tecnica do cadastro). "
                    "Omita para buscar só documentos gerais e todas as fichas.")] = None,
) -> dict[str, Any]:
    """Busca na documentação técnica da Construtora Apollo: fichas técnicas de manutenção por modelo,
    catálogo de alarmes TL, procedimento de segurança PRO-SEG-001 e política de manutenção POL-MAN-001.

    Use para REGRAS e CONHECIMENTO: significado de alarmes, sequência de diagnóstico, peças associadas a um
    sistema, critérios de bloqueio, matriz de prioridade, regra de falha recorrente e devolução de OS.
    Faça UMA pergunta por assunto, citando o sintoma, o sistema ou o código de alarme. Nunca combine
    assuntos numa só pergunta: cada busca devolve só 3 trechos. Cada trecho traz a seção inteira, inclusive
    a linha "Peças associadas" com os códigos APL. NÃO use para fatos operacionais."""
    def executar() -> dict[str, Any]:
        filtro = {"aplicacao": {"$in": [ficha_tecnica, "geral"]}} if ficha_tecnica else None
        resultado = colecao.query(query_embeddings=[list(_embedding_consulta(pergunta.strip()))], n_results=3,
                                  where=filtro, include=["documents", "metadatas", "distances"])
        evidencias = [
            {"referencia": meta["referencia"], "documento": meta["titulo_documento"],
             "secao": meta["titulo"], "trecho": _encurtar(texto)}
            for texto, meta in zip(resultado["documents"][0], resultado["metadatas"][0])
        ]
        if not evidencias:
            return _nao_encontrado("Nenhum trecho da documentação corresponde à pergunta.")
        return _sucesso(pergunta=pergunta, filtro=ficha_tecnica or "todos os documentos", evidencias=evidencias)
    return _executar("buscar_documentacao_tecnica", ("evidencias",), executar)


if PERFIL == "completo":
    mcp.tool()(buscar_documentacao_tecnica)


@mcp.tool()
def consultar_equipamento(
    equipamento_id: Annotated[str | None, Field(
        description="Prefixo do equipamento na frota, ex.: EH-07, CR-04, CB-12.")] = None,
    obra_id: Annotated[Literal[OBRAS] | None, Field(description="Obra, para listar candidatos.")] = None,
    tipo: Annotated[Literal[TIPOS] | None, Field(description="Tipo de equipamento, para listar candidatos.")] = None,
) -> dict[str, Any]:
    """Consulta o cadastro da frota. Com equipamento_id, confirma se o equipamento existe e devolve tipo,
    modelo, obra, status, criticidade, horímetro e ficha técnica. Sem equipamento_id, lista os candidatos de
    uma obra e/ou tipo. Candidatos NÃO autorizam escolher um equipamento por aproximação (POL-MAN-001)."""
    def executar() -> dict[str, Any]:
        if not (equipamento_id or obra_id or tipo):
            return _argumento_invalido("Informe equipamento_id, ou obra_id e/ou tipo para listar candidatos.")
        candidatos = TABELAS["equipamentos"]
        if obra_id:
            candidatos = candidatos[candidatos["obra_id"] == obra_id]
        if tipo:
            candidatos = candidatos[candidatos["tipo"] == tipo]
        lista = [_resumo_equipamento(l) for _, l in candidatos.iterrows()] if (obra_id or tipo) else []
        if equipamento_id:
            linha = _localizar_equipamento(equipamento_id)
            if linha is not None:
                return _sucesso(equipamento=_resumo_equipamento(linha))
            return _nao_encontrado(
                f"O equipamento {_normalizar_id_equipamento(equipamento_id)} não existe no cadastro.",
                candidatos=lista,
                observacao="Candidatos apenas atendem aos filtros informados; não indicam qual é o "
                           "equipamento da ordem de serviço.")
        if not lista:
            return _nao_encontrado("Nenhum equipamento atende aos filtros informados.")
        return _sucesso(quantidade=len(lista), candidatos=lista)
    return _executar("consultar_equipamento", (), executar)


@mcp.tool()
def consultar_historico_manutencao(
    equipamento_id: Annotated[str, Field(min_length=2, max_length=12, description="Prefixo do equipamento.")],
    sistema: Annotated[Literal[SISTEMAS] | None, Field(description="Filtra a lista de ocorrências.")] = None,
    janela_dias: Annotated[int, Field(ge=1, le=365, description="Janela da lista de ocorrências.")] = 180,
) -> dict[str, Any]:
    """Histórico de manutenções CORRETIVAS do equipamento: ordens anteriores, sistema, alarme, serviço
    executado e peças trocadas, além da contagem de corretivas por sistema nos últimos 90 dias. Use para
    verificar recorrência e o que já foi tentado. A ordem em triagem não está incluída nas contagens."""
    def executar() -> dict[str, Any]:
        linha = _localizar_equipamento(equipamento_id)
        if linha is None:
            return _nao_encontrado(f"O equipamento {equipamento_id} não existe no cadastro.")
        hist = TABELAS["historico_manutencao"]
        hist = hist[hist["equipamento_id"] == linha["equipamento_id"]].copy()
        hist["data"] = pd.to_datetime(hist["data_abertura"])
        recentes = hist[hist["data"] >= DATA_REFERENCIA - pd.Timedelta(days=JANELA_RECORRENCIA_DIAS)]
        na_janela = hist[hist["data"] >= DATA_REFERENCIA - pd.Timedelta(days=int(janela_dias))]
        if sistema:
            na_janela = na_janela[na_janela["sistema"] == sistema]
        ocorrencias = [
            {"ordem_id": l["ordem_id"], "data_abertura": l["data_abertura"], "sistema": l["sistema"],
             "codigo_alarme": l["codigo_alarme"] or None, "servico_executado": l["descricao_servico"],
             "pecas_utilizadas": [p for p in l["pecas_utilizadas"].split(";") if p],
             "horas_parado": int(l["horas_parado"]) if l["horas_parado"] else "em andamento"}
            for _, l in na_janela.sort_values("data", ascending=False).iterrows()
        ]
        return _sucesso(
            equipamento_id=linha["equipamento_id"], janela_dias=int(janela_dias),
            filtro_sistema=sistema or "todos", total_ocorrencias=len(ocorrencias), ocorrencias=ocorrencias,
            corretivas_por_sistema_ultimos_90_dias=recentes.groupby("sistema").size().astype(int).to_dict(),
            observacao="Contagens e lista NÃO incluem a ordem de serviço que está em triagem.")
    return _executar("consultar_historico_manutencao", ("ocorrencias", "corretivas_por_sistema_ultimos_90_dias"),
                     executar)


@mcp.tool()
def consultar_plano_preventiva(
    equipamento_id: Annotated[str, Field(min_length=2, max_length=12, description="Prefixo do equipamento.")],
) -> dict[str, Any]:
    """Situação das revisões preventivas do equipamento pelo horímetro atual: em_dia, proxima ou vencida,
    com horas desde a última execução. Use quando o sintoma puder estar ligado a manutenção atrasada."""
    def executar() -> dict[str, Any]:
        linha = _localizar_equipamento(equipamento_id)
        if linha is None:
            return _nao_encontrado(f"O equipamento {equipamento_id} não existe no cadastro.")
        eq_id, horimetro = linha["equipamento_id"], int(linha["horimetro_atual"])
        plano = TABELAS["plano_preventiva"]
        plano = plano[plano["ficha_tecnica"] == linha["ficha_tecnica"]]
        registro = TABELAS["registro_preventivas"]
        registro = registro[registro["equipamento_id"] == eq_id].set_index("revisao")
        revisoes = []
        for _, rev in plano.iterrows():
            intervalo = int(rev["intervalo_horas"])
            if rev["revisao"] in registro.index:
                execucao = registro.loc[rev["revisao"]]
                horas_desde = horimetro - int(execucao["horimetro_execucao"])
                fracao = horas_desde / intervalo
                situacao = ("vencida" if fracao > 1 + TOLERANCIA_PREVENTIVA
                            else "proxima" if fracao >= 0.9 else "em_dia")
                revisoes.append({
                    "revisao": rev["revisao"], "intervalo_horas": intervalo,
                    "ultima_execucao_horimetro": int(execucao["horimetro_execucao"]),
                    "ultima_execucao_data": execucao["data_execucao"], "horas_desde_ultima": horas_desde,
                    "percentual_do_intervalo": round(100 * fracao), "situacao": situacao})
            else:
                revisoes.append({"revisao": rev["revisao"], "intervalo_horas": intervalo,
                                 "ultima_execucao_horimetro": None,
                                 "situacao": "nao_atingida" if horimetro < intervalo else "vencida"})
        return _sucesso(
            equipamento_id=eq_id, ficha_tecnica=linha["ficha_tecnica"], horimetro_atual=horimetro,
            revisoes=revisoes,
            revisoes_vencidas=[r["revisao"] for r in revisoes if r["situacao"] == "vencida"],
            revisoes_proximas=[r["revisao"] for r in revisoes if r["situacao"] == "proxima"],
            criterio="vencida acima de 110% do intervalo; próxima a partir de 90%")
    return _executar("consultar_plano_preventiva", ("revisoes", "revisoes_vencidas"), executar)


@mcp.tool()
def consultar_estoque_pecas(
    codigos_peca: Annotated[list[Annotated[str, Field(pattern="^APL-[0-9]{5}$")]] | None, Field(
        min_length=1, max_length=8, description="Códigos internos no formato APL-00000.")] = None,
    ficha_tecnica: Annotated[Literal[FICHAS] | None, Field(
        description="Alternativa aos códigos: devolve todas as peças associadas à ficha técnica.")] = None,
    obra_id: Annotated[Literal[OBRAS] | None, Field(
        description="Obra do equipamento, para destacar o almoxarifado local.")] = None,
) -> dict[str, Any]:
    """Saldo de peças por almoxarifado, destacando o almoxarifado da obra, e prazo de reposição. Informe os
    códigos APL das peças ou a ficha técnica do equipamento (todas as peças associadas a ela)."""
    def executar() -> dict[str, Any]:
        estoque = TABELAS["estoque_pecas"]
        if codigos_peca:
            codigos = list(dict.fromkeys(c.strip().upper() for c in codigos_peca))
        elif ficha_tecnica:
            codigos = list(dict.fromkeys(estoque.loc[estoque["ficha_tecnica"] == ficha_tecnica, "codigo_peca"]))
        else:
            return _argumento_invalido("Informe codigos_peca ou ficha_tecnica.")
        obras = TABELAS["obras"].set_index("obra_id")
        almox_obra = obras.loc[obra_id, "almoxarifado_id"] if obra_id else None
        pecas, nao_encontrados = [], []
        for codigo in codigos:
            linhas = estoque[estoque["codigo_peca"] == codigo]
            if linhas.empty:
                nao_encontrados.append(codigo)
                continue
            saldo_total = int(linhas["saldo"].sum())
            saldo_obra = int(linhas.loc[linhas["almoxarifado_id"] == almox_obra, "saldo"].sum()) if almox_obra else None
            if almox_obra and saldo_obra > 0:
                situacao = "disponivel_na_obra"
            elif saldo_total > 0:
                situacao = "disponivel_em_outro_almoxarifado" if almox_obra else "disponivel"
            else:
                situacao = "sem_estoque"
            pecas.append({
                "codigo_peca": codigo, "descricao": linhas.iloc[0]["descricao"],
                "ref_fabricante": linhas.iloc[0]["ref_fabricante"] or None, "situacao": situacao,
                "saldo_total": saldo_total, "saldo_no_almoxarifado_da_obra": saldo_obra,
                "almoxarifados_com_saldo": [{"almoxarifado_id": l["almoxarifado_id"], "saldo": int(l["saldo"])}
                                            for _, l in linhas[linhas["saldo"] > 0].iterrows()],
                "prazo_reposicao_dias": int(linhas.iloc[0]["prazo_reposicao_dias"])})
        if not pecas:
            return _nao_encontrado("Nenhum dos códigos informados existe no estoque.",
                                   codigos_nao_encontrados=nao_encontrados)
        return _sucesso(obra_id=obra_id, almoxarifado_da_obra=almox_obra, pecas=pecas,
                        codigos_nao_encontrados=nao_encontrados)
    return _executar("consultar_estoque_pecas", ("pecas",), executar)


@mcp.tool()
def consultar_frota_disponivel(
    tipo: Annotated[Literal[TIPOS], Field(description="Tipo de equipamento.")],
    obra_id: Annotated[Literal[OBRAS], Field(description="Obra de referência.")],
) -> dict[str, Any]:
    """Lista equipamentos do mesmo tipo com status Disponível em qualquer obra, para avaliar substituto."""
    def executar() -> dict[str, Any]:
        eq = TABELAS["equipamentos"]
        disponiveis = eq[(eq["tipo"] == tipo) & (eq["status_operacional"] == "Disponível")].copy()
        disponiveis["na_mesma_obra"] = disponiveis["obra_id"] == obra_id
        lista = [{"equipamento_id": l["equipamento_id"], "modelo": l["modelo"], "obra_id": l["obra_id"],
                  "na_mesma_obra": bool(l["na_mesma_obra"]), "horimetro_atual": int(l["horimetro_atual"])}
                 for _, l in disponiveis.sort_values("na_mesma_obra", ascending=False).iterrows()]
        return _sucesso(tipo=tipo, obra_de_referencia=obra_id, quantidade=len(lista), equipamentos=lista,
                        mensagem=None if lista else "Nenhum equipamento deste tipo está disponível em nenhuma obra.")
    return _executar("consultar_frota_disponivel", ("equipamentos",), executar)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

SERVIDOR_MCP.write_text(CODIGO_SERVIDOR_MCP, encoding="utf-8")

AMBIENTE_SERVIDOR_MCP = {
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "APOLLO_MCP_PERFIL": "completo",
    "APOLLO_DADOS": str(PASTA_DADOS),
    "APOLLO_CHROMA_DIR": str(Path(DIRETORIO_CHROMA).resolve()),
    "APOLLO_CHROMA_COLECAO": NOME_COLECAO,
    "APOLLO_DATA_REFERENCIA": DATA_REFERENCIA.isoformat(),
}

print(f"Servidor MCP gravado em {SERVIDOR_MCP.resolve()} ({len(CODIGO_SERVIDOR_MCP.splitlines())} linhas)")

## 7.2 Cliente MCP

No cliente, uma task assíncrona é dona da sessão MCP e as chamadas chegam a ela por uma fila, o que evita problemas de ciclo de vida no Colab. Antes de cada chamada, os argumentos são validados contra o schema publicado pelo servidor.

In [ ]:
class ClienteMCP:
    """Cliente MCP para Colab/Jupyter.

    Uma única task é dona do ciclo de vida da sessão MCP. As outras células pedem
    chamadas por uma fila assíncrona, para que a sessão seja aberta e fechada pela mesma task.
    """

    def __init__(self, script_do_servidor: str | Path) -> None:
        self.script = Path(script_do_servidor).resolve()
        self._client: Any = None
        self._worker_task: asyncio.Task | None = None
        self._fila: asyncio.Queue | None = None
        self._ready: asyncio.Future | None = None
        self._tools: list[Any] = []
        self._catalogo: dict[str, Any] = {}
        self.servidor = ""
        self.versao_do_protocolo = ""
        self._errlog_path = Path("mcp_server_stderr.log").resolve()

    def _garantir_ativo(self) -> None:
        if self._worker_task is None or self._fila is None:
            raise RuntimeError("Cliente MCP não conectado.")
        if self._worker_task.done():
            raise RuntimeError("A sessão MCP já foi encerrada.")

    async def conectar(self) -> "ClienteMCP":
        """Inicia a task dona da sessão e aguarda o handshake e o discovery."""
        if self._worker_task is not None and not self._worker_task.done():
            return self
        self._fila = asyncio.Queue()
        self._ready = asyncio.get_running_loop().create_future()
        self._worker_task = asyncio.create_task(self._worker(), name="apollo-mcp-client-owner")
        await self._ready
        return self

    async def _worker(self) -> None:
        """Abre, usa e fecha a sessão MCP dentro da mesma task."""
        from mcp import Client, StdioServerParameters
        from mcp.client.stdio import stdio_client

        close_future: asyncio.Future | None = None
        parametros = StdioServerParameters(
            command=sys.executable,
            args=[str(self.script)],
            env=AMBIENTE_SERVIDOR_MCP,
            cwd=str(Path.cwd()),
        )
        try:
            with open(self._errlog_path, "a", encoding="utf-8", buffering=1) as errlog:
                async with Client(stdio_client(parametros, errlog=errlog)) as client:
                    self._client = client
                    info = client.server_info
                    self.servidor = info.name if info is not None else "servidor sem identidade"
                    self.versao_do_protocolo = str(client.protocol_version)
                    await self._descobrir_ferramentas(client)
                    self._ready.set_result(True)

                    while True:
                        comando = await self._fila.get()
                        futuro: asyncio.Future = comando["future"]
                        if comando["op"] == "close":
                            close_future = futuro
                            break
                        try:
                            resultado = await client.call_tool(comando["nome"], comando["argumentos"])
                            futuro.set_result(resultado)
                        except Exception as erro:
                            futuro.set_exception(erro)
            if close_future is not None and not close_future.done():
                close_future.set_result(None)
        except Exception as erro:
            if not self._ready.done():
                self._ready.set_exception(erro)
            if close_future is not None and not close_future.done():
                close_future.set_exception(erro)
            raise
        finally:
            self._client = None

    async def encerrar(self) -> None:
        """Pede à task dona para fechar a sessão MCP."""
        if self._worker_task is None:
            return
        task = self._worker_task
        if not task.done():
            futuro = asyncio.get_running_loop().create_future()
            await self._fila.put({"op": "close", "future": futuro})
            await futuro
        await task
        self._worker_task = self._fila = self._ready = None
        self._tools, self._catalogo = [], {}

    async def _descobrir_ferramentas(self, client: Any) -> None:
        ferramentas: list[Any] = []
        cursor: str | None = None
        while True:
            pagina = await client.list_tools(cursor=cursor)
            ferramentas.extend(pagina.tools)
            cursor = pagina.next_cursor
            if cursor is None:
                break
        self._tools = ferramentas
        self._catalogo = {tool.name: tool for tool in ferramentas}

    def listar_ferramentas(self) -> list[dict[str, Any]]:
        return [{"nome": t.name, "descricao": (t.description or "").strip(), "input_schema": t.input_schema}
                for t in self._tools]

    def _validar_argumentos(self, nome: str, argumentos: dict[str, Any]) -> None:
        """Valida a chamada contra o schema publicado pelo servidor."""
        tool = self._catalogo.get(nome)
        if tool is None:
            raise ValueError(f"Ferramenta não publicada pelo servidor MCP: {nome}")
        erros = sorted(Draft202012Validator(tool.input_schema).iter_errors(argumentos),
                       key=lambda erro: list(erro.path))
        if erros:
            mensagens = [f"{'.'.join(str(p) for p in e.path) or 'argumentos'}: {e.message}" for e in erros[:3]]
            raise ValueError(f"Argumentos inválidos para {nome}: " + "; ".join(mensagens))

    @staticmethod
    def _texto_do_resultado(resultado: Any) -> str:
        return "\n".join(getattr(b, "text", "") for b in getattr(resultado, "content", []) or [])

    async def chamar(self, nome: str, argumentos: dict[str, Any]) -> dict[str, Any]:
        """Valida o contrato e executa uma ferramenta MCP."""
        self._validar_argumentos(nome, argumentos)
        self._garantir_ativo()
        futuro = asyncio.get_running_loop().create_future()
        await self._fila.put({"op": "call_tool", "nome": nome, "argumentos": argumentos, "future": futuro})
        resultado = await futuro
        if resultado.is_error:
            raise RuntimeError(self._texto_do_resultado(resultado) or f"A ferramenta {nome} retornou erro.")
        payload = resultado.structured_content
        if isinstance(payload, dict):
            return payload
        return {"status": "success", "content": self._texto_do_resultado(resultado)}


print("ClienteMCP definido.")

In [ ]:
cliente_mcp = ClienteMCP(SERVIDOR_MCP)
await cliente_mcp.conectar()

print("Servidor:", cliente_mcp.servidor, "· protocolo", cliente_mcp.versao_do_protocolo)
print("\nFerramentas descobertas:")
for contrato in cliente_mcp.listar_ferramentas():
    print(f"  {contrato['nome']:<32} {list(contrato['input_schema'].get('properties', {}))}")

---
# 8. Parte 1 — O agente em LangGraph

A Parte 1 refatora o agente da Entrega 1 com LangGraph. Ele continua sendo **um único agente**: os nodes do grafo são responsabilidades dele, não agentes separados.

| Node | Natureza | Responsabilidade |
|---|---|---|
| `perceive` | determinístico | valida a OS e monta o contexto inicial |
| `model` | LLM | decide quais ferramentas consultar |
| `tools` | ToolNode + MCP | executa as ferramentas e devolve as observações |
| `finalize` | LLM | gera o parecer estruturado |
| `validate` | determinístico | confere o parecer contra as evidências e aplica o guardrail |
| `action` | determinístico | grava o parecer na fila de manutenção |

## 8.1 Ferramentas MCP como `StructuredTool`

No adaptador, cada contrato descoberto no servidor MCP vira uma `StructuredTool` do LangChain, que chama o `ClienteMCP` quando o modelo pede a ferramenta. O conteúdo vai para o modelo como JSON e o payload completo fica em `artifact`, para o node `tools` registrar as evidências no estado.

O comportamento da Entrega 1 foi mantido: argumento inválido ou falha na ferramenta voltam ao modelo como `invalid_arguments` ou `error`, e a triagem segue.

In [ ]:
LOG_FERRAMENTAS: list[dict[str, Any]] = []    # detalhes técnicos que não vão para o modelo


def adaptar_tools_mcp_para_langchain(cliente: ClienteMCP) -> list[StructuredTool]:
    """Cria tools LangChain a partir do discovery MCP."""
    tools_langchain: list[StructuredTool] = []
    for contrato in cliente.listar_ferramentas():
        nome = contrato["nome"]

        def fabricar_coroutine(nome_da_tool: str):
            async def executar_via_mcp(**argumentos: Any):
                argumentos = {k: v for k, v in argumentos.items() if v is not None}
                try:
                    payload = await cliente.chamar(nome_da_tool, argumentos)
                except ValueError as erro:
                    payload = {"status": "invalid_arguments", "mensagem": str(erro)}
                except Exception as erro:
                    LOG_FERRAMENTAS.append({"ferramenta": nome_da_tool, "erro": f"{type(erro).__name__}: {erro}"})
                    payload = {"status": "error", "mensagem": "Falha ao executar a ferramenta. Prossiga com as "
                                                              "demais fontes e registre esta verificação como pendência."}
                # content → vai para o modelo · artifact → fica para o estado do grafo
                return json.dumps(payload, ensure_ascii=False, default=str), payload
            return executar_via_mcp

        tools_langchain.append(StructuredTool.from_function(
            coroutine=fabricar_coroutine(nome),
            name=nome,
            description=contrato["descricao"],
            args_schema=contrato["input_schema"],
            infer_schema=False,
            response_format="content_and_artifact",
        ))
    return tools_langchain


tools_langchain = adaptar_tools_mcp_para_langchain(cliente_mcp)
tool_node_base = ToolNode(tools_langchain, handle_tool_errors=False)

print("Tools LangChain criadas a partir do MCP:")
for tool in tools_langchain:
    print(f"  {tool.name:<32} {list(tool.args.keys())}")

## 8.2 Lógica de negócio preservada da Entrega 1

Percepção, instruções, schema do parecer, validação, guardrail, parecer de contingência e fila de manutenção vêm da Entrega 1 sem mudança de regra. Só mudaram duas coisas:

- as instruções não falam mais da ferramenta terminal `emitir_parecer`: o modelo para de pedir ferramentas quando tem evidência suficiente, e o parecer é gerado no node `finalize`;
- a fila ganhou a coluna `orquestracao`, para registrar se o parecer veio da Parte 1 ou da Parte 2.

In [ ]:
SISTEMAS = ["Hidráulico", "Motor", "Arrefecimento", "Elétrico", "Trem de força", "Freios",
            "Direção", "Suspensão", "Material rodante", "Estrutura e implementos"]
MAX_RODADAS = 6                  # chamadas ao modelo por triagem, incluindo a finalização
MAX_CHAMADAS_FERRAMENTA = 12      # chamadas de ferramenta por triagem
ORDENS = TABELAS["ordens_servico"].set_index("os_id", drop=False)

# ----------------------------------------------------------------- percepção (Entrega 1)
GATILHOS_SEGURANCA: dict[str, str] = {
    "falha ou anormalidade em freio": r"\bfrei(o|os|ar|ando|ada)\b|\bfrenagem\b",
    "falha na direção": r"\bdirecao\b|\bvolante (duro|pesado|com folga|travad)",
    "fumaça na cabine / cheiro de queimado": r"fumaca (na|dentro da) cabine|cheiro de queimad",
    "indício de incêndio": r"\bincendio\b|\bfogo\b|\bchamas?\b",
    "vazamento de combustível": r"vazamento de (combustivel|diesel|oleo diesel)",
    "vazamento hidráulico com jato sob pressão": r"\bjato\b",
    "trinca estrutural": r"\btrinca",
}
ALARMES_NIVEL_3 = {"TL-120", "TL-331", "TL-350"}       # CAT-ALM-001, trava fixa
RE_ALARME = re.compile(r"\bTL-\d{3}\b", re.IGNORECASE)


def _sem_acentos(texto: str) -> str:
    decomposto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in decomposto if not unicodedata.combining(c))


def _normalizar_id_equipamento(valor: str) -> str:
    texto = re.sub(r"[\s_]+", "-", valor.strip().upper())
    return f"{texto[:2]}-{texto[2:]}" if re.fullmatch(r"[A-Z]{2}\d{2}", texto) else texto


def _localizar_equipamento(equipamento_id: str) -> pd.Series | None:
    eq = TABELAS["equipamentos"]
    linha = eq[eq["equipamento_id"] == _normalizar_id_equipamento(equipamento_id)]
    return None if linha.empty else linha.iloc[0]


def _resumo_equipamento(linha: pd.Series) -> dict[str, Any]:
    return {"equipamento_id": linha["equipamento_id"], "tipo": linha["tipo"], "modelo": linha["modelo"],
            "obra_id": linha["obra_id"], "status_operacional": linha["status_operacional"],
            "criticidade": linha["criticidade"], "horimetro_atual": int(linha["horimetro_atual"]),
            "ficha_tecnica": linha["ficha_tecnica"]}


@dataclass
class Percepcao:
    os_id: str
    valida: bool
    motivo_rejeicao: str | None
    contexto: dict[str, Any]
    equipamento: dict[str, Any] | None
    codigos_alarme: list[str]
    gatilhos: list[str]


def perceber(os_linha: pd.Series) -> Percepcao:
    """Valida a OS e monta o contexto inicial do agente."""
    os_id = str(os_linha.get("os_id", "")).strip() or "SEM-ID"
    relato = str(os_linha.get("relato", "")).strip()
    obra_id = str(os_linha.get("obra_id", "")).strip()
    obras = TABELAS["obras"].set_index("obra_id")
    motivo = None
    if not relato:
        motivo = "OS sem relato do problema."
    elif len(relato) < 15:
        motivo = "Relato curto demais para qualquer triagem."
    elif obra_id not in obras.index:
        motivo = f"Obra '{obra_id}' inexistente no cadastro."
    if motivo:
        return Percepcao(os_id, False, motivo, {}, None, [], [])

    codigos = sorted({c.upper() for c in RE_ALARME.findall(relato)})
    texto = _sem_acentos(relato)
    gatilhos = [nome for nome, padrao in GATILHOS_SEGURANCA.items() if re.search(padrao, texto)]
    gatilhos += [f"alarme de nível 3 ({c})" for c in codigos if c in ALARMES_NIVEL_3]
    informado = str(os_linha.get("equipamento_informado", "")).strip()
    linha_eq = _localizar_equipamento(informado) if informado else None
    equipamento = _resumo_equipamento(linha_eq) if linha_eq is not None else None
    contexto = {
        "os_id": os_id, "data_abertura": os_linha.get("data_abertura"),
        "obra": {"obra_id": obra_id, "nome": obras.loc[obra_id, "nome"]},
        "aberta_por": os_linha.get("aberta_por"), "relato": relato,
        "equipamento_informado": informado or "(não informado)",
        "equipamento_no_cadastro": equipamento or (
            "NÃO ENCONTRADO: o identificador informado não existe no cadastro da frota."
            if informado else "NÃO INFORMADO"),
        "codigos_de_alarme_no_relato": codigos,
        "gatilhos_de_seguranca_detectados": gatilhos,
    }
    if equipamento and equipamento["obra_id"] != obra_id:
        contexto["divergencia"] = (f"O cadastro situa {equipamento['equipamento_id']} na obra "
                                   f"{equipamento['obra_id']}, mas a OS foi aberta na {obra_id}.")
    return Percepcao(os_id, True, None, contexto, equipamento, codigos, gatilhos)


# ----------------------------------------------------------------- instruções
INSTRUCOES_SISTEMA = f"""Você é o agente de triagem de ordens de serviço (OS) de manutenção da Construtora Apollo S.A.
Sua tarefa: investigar a OS recebida e reunir as evidências para o parecer que orienta o planejador de manutenção.
Você RECOMENDA; não executa reparos, não bloqueia nem libera equipamentos. A data de hoje é {DATA_REFERENCIA:%d/%m/%Y}.

FONTES
- Regras e conhecimento técnico (significado de alarmes, diagnóstico, peças associadas, critérios de bloqueio,
  matriz de prioridade, falha recorrente, devolução de OS): buscar_documentacao_tecnica.
- Fatos operacionais: consultar_equipamento, consultar_historico_manutencao, consultar_plano_preventiva,
  consultar_estoque_pecas, consultar_frota_disponivel.

PROCESSO
1. Equipamento: se o contexto não trouxer o equipamento no cadastro, use consultar_equipamento. Sem identificação
   inequívoca, a OS deve ser DEVOLVIDA (POL-MAN-001 § 3). Nunca escolha um equipamento por aproximação.
2. Relato: se ele não permitir direcionar a investigação, consulte a ficha técnica para saber o que perguntar.
3. Investigação: consulte a documentação sobre o sintoma ou alarme (pergunta específica, informando ficha_tecnica),
   o histórico e a preventiva quando o sintoma puder ter relação com ela. Verifique no estoque as peças APL que a
   documentação associar ao problema.
4. Prioridade: recupere a matriz de prioridade e, se houver repetição, a regra de falha recorrente da POL-MAN-001.
   Para equipamento de criticidade A ou parada provável, verifique substituto com consultar_frota_disponivel.
5. Segurança: se houver gatilho de segurança no contexto ou na documentação, recupere o PRO-SEG-001.

REGRAS
- Peça na MESMA rodada todas as consultas independentes entre si.
- Na documentação, UMA pergunta por assunto. Cada trecho traz a seção inteira, com a linha "Peças associadas".
- Nunca afirme fato operacional sem a ferramenta correspondente.
- Quando tiver evidência suficiente para concluir ou para devolver a OS, pare de pedir ferramentas:
  o parecer é gerado na etapa seguinte.
- Limites desta triagem: {MAX_RODADAS} chamadas ao modelo e {MAX_CHAMADAS_FERRAMENTA} chamadas de ferramenta."""

PROMPT_FINAL = """Gere agora o parecer da triagem usando somente a OS e as evidências presentes no histórico.
- Sem histórico consultado, falha_recorrente = nao_verificado; sem preventiva, situacao_preventiva = nao_verificada;
  sem estoque, disponibilidade = nao_verificada.
- fontes_tecnicas: apenas referências efetivamente recuperadas, no formato exato devolvido (ex.: "FTM-CAT320 § 5.2").
- Equipamento não identificado ou relato insuficiente: status_triagem = devolvida, com perguntas_ao_solicitante.
- Se a frota foi consultada e há substituto disponível em qualquer obra, informe-o em equipamento_substituto.
- Não liste em pecas_provaveis componente já trocado em corretiva recente sem diagnóstico que justifique nova troca.
- Registre em pendencias o que não foi possível verificar."""

# ----------------------------------------------------------------- parecer (Entrega 1)
PARECER_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "status_triagem": {"type": "string", "enum": ["concluida", "devolvida"]},
        "equipamento_id": {"type": "string", "description": "Prefixo do equipamento no cadastro, ou NAO_IDENTIFICADO."},
        "sistema_afetado": {"type": "string", "enum": SISTEMAS + ["Indeterminado"]},
        "diagnostico_provavel": {"type": "string", "description": "Hipótese principal, em até três frases."},
        "confianca": {"type": "string", "enum": ["alta", "media", "baixa"]},
        "procedimento_recomendado": {"type": "array", "maxItems": 8, "items": {"type": "string"}},
        "pecas_provaveis": {"type": "array", "maxItems": 8, "items": {
            "type": "object",
            "properties": {
                "codigo_peca": {"type": "string", "pattern": "^APL-[0-9]{5}$"},
                "descricao": {"type": "string"},
                "disponibilidade": {"type": "string", "enum": ["na_obra", "em_outro_almoxarifado",
                                                              "sem_estoque", "nao_verificada"]}},
            "required": ["codigo_peca", "descricao", "disponibilidade"],
            "additionalProperties": False}},
        "falha_recorrente": {"type": "string", "enum": ["sim", "nao", "nao_verificado"]},
        "situacao_preventiva": {"type": "string", "enum": ["em_dia", "proxima", "vencida", "nao_verificada"]},
        "prioridade": {"type": "string", "enum": ["P1", "P2", "P3", "P4"]},
        "recomendacao_operacional": {"type": "string",
                                     "enum": ["liberar", "liberar_com_restricao", "bloquear_ate_inspecao"]},
        "equipamento_substituto": {"type": "string"},
        "justificativa": {"type": "string"},
        "fontes_tecnicas": {"type": "array", "items": {"type": "string"}},
        "pendencias": {"type": "array", "items": {"type": "string"}},
        "perguntas_ao_solicitante": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["status_triagem", "equipamento_id", "sistema_afetado", "diagnostico_provavel", "confianca",
                 "procedimento_recomendado", "pecas_provaveis", "falha_recorrente", "situacao_preventiva",
                 "prioridade", "recomendacao_operacional", "justificativa", "fontes_tecnicas",
                 "pendencias", "perguntas_ao_solicitante"],
    "additionalProperties": False,
}
VALIDADOR_PARECER = Draft202012Validator(PARECER_SCHEMA)


def validar_parecer(parecer: dict[str, Any], percepcao: dict[str, Any], referencias: set[str],
                    ferramentas_ok: set[str], pecas_consultadas: set[str]) -> list[str]:
    """Confere o parecer contra o schema e contra o que foi efetivamente observado (Entrega 1)."""
    problemas = [f"{'.'.join(str(p) for p in e.path) or 'parecer'}: {e.message}"
                 for e in VALIDADOR_PARECER.iter_errors(parecer)][:3]
    if problemas:
        return problemas
    ids_cadastro = set(TABELAS["equipamentos"]["equipamento_id"])
    eq_percebido, eq_parecer = percepcao.get("equipamento"), parecer["equipamento_id"]
    fontes_invalidas = [f for f in parecer["fontes_tecnicas"] if f not in referencias]
    if fontes_invalidas:
        problemas.append(f"fontes_tecnicas citadas sem terem sido recuperadas: {fontes_invalidas}")
    if parecer["falha_recorrente"] != "nao_verificado" and "consultar_historico_manutencao" not in ferramentas_ok:
        problemas.append("falha_recorrente afirmada sem consulta bem-sucedida ao histórico")
    if parecer["situacao_preventiva"] != "nao_verificada" and "consultar_plano_preventiva" not in ferramentas_ok:
        problemas.append("situacao_preventiva afirmada sem consulta bem-sucedida ao plano de preventiva")
    for peca in parecer["pecas_provaveis"]:
        if peca["disponibilidade"] != "nao_verificada" and peca["codigo_peca"] not in pecas_consultadas:
            problemas.append(f"disponibilidade de {peca['codigo_peca']} afirmada sem consulta ao estoque")
    if eq_parecer != "NAO_IDENTIFICADO" and eq_parecer not in ids_cadastro:
        problemas.append(f"equipamento_id {eq_parecer} não existe no cadastro")
    if eq_percebido and eq_parecer != eq_percebido["equipamento_id"]:
        problemas.append(f"equipamento_id deveria ser {eq_percebido['equipamento_id']} (identificado na OS)")
    if parecer["status_triagem"] == "devolvida" and not parecer["perguntas_ao_solicitante"]:
        problemas.append("OS devolvida sem perguntas_ao_solicitante")
    if parecer["status_triagem"] == "concluida":
        if eq_parecer == "NAO_IDENTIFICADO":
            problemas.append("triagem concluída sem equipamento identificado: deveria ser devolvida")
        if not parecer["fontes_tecnicas"]:
            problemas.append("triagem concluída sem nenhuma fonte técnica que sustente o diagnóstico")
    return problemas


def aplicar_guardrail(parecer: dict[str, Any], gatilhos: list[str]) -> list[str]:
    """Gatilho de segurança impõe bloqueio e P1. Devolve as sobrescritas feitas."""
    sobrescritas = []
    if gatilhos:
        if parecer.get("recomendacao_operacional") != "bloquear_ate_inspecao":
            sobrescritas.append(f"recomendacao_operacional: {parecer.get('recomendacao_operacional')} "
                                "→ bloquear_ate_inspecao (gatilho de segurança, PRO-SEG-001 § 3)")
            parecer["recomendacao_operacional"] = "bloquear_ate_inspecao"
        if parecer.get("prioridade") != "P1":
            sobrescritas.append(f"prioridade: {parecer.get('prioridade')} → P1 (POL-MAN-001 § 4)")
            parecer["prioridade"] = "P1"
    return sobrescritas


def parecer_incompleto(percepcao: dict[str, Any], referencias: set[str], motivo: str) -> dict[str, Any]:
    """Parecer de contingência quando a triagem não pode ser concluída pelo modelo."""
    eq, gatilhos = percepcao.get("equipamento"), percepcao.get("gatilhos", [])
    return {
        "status_triagem": "incompleta",
        "equipamento_id": eq["equipamento_id"] if eq else "NAO_IDENTIFICADO",
        "sistema_afetado": "Indeterminado",
        "diagnostico_provavel": "Não determinado: a triagem automática foi interrompida.",
        "confianca": "baixa", "procedimento_recomendado": [], "pecas_provaveis": [],
        "falha_recorrente": "nao_verificado", "situacao_preventiva": "nao_verificada",
        "prioridade": "P1" if gatilhos else "P2",
        "recomendacao_operacional": "bloquear_ate_inspecao" if gatilhos else "liberar_com_restricao",
        "equipamento_substituto": "",
        "justificativa": f"Triagem interrompida: {motivo.rstrip('.')}. Realizar a triagem manualmente.",
        "fontes_tecnicas": sorted(referencias), "pendencias": [motivo, "Triagem manual obrigatória."],
        "perguntas_ao_solicitante": [],
    }


# ----------------------------------------------------------------- fila (Entrega 1)
ARQUIVO_FILA = Path("fila_manutencao.csv")
FILA: dict[tuple[str, str], dict[str, Any]] = {}


def agir(registro: dict[str, Any]) -> None:
    """Grava o parecer na fila de manutenção; uma linha por OS e por orquestração."""
    p = registro.get("parecer") or {}
    FILA[(registro["os_id"], registro["orquestracao"])] = {
        "os_id": registro["os_id"], "orquestracao": registro["orquestracao"],
        "data_triagem": DATA_REFERENCIA.isoformat(), "status_triagem": registro["status_triagem"],
        "equipamento_id": registro["equipamento_id"], "prioridade": registro["prioridade"],
        "recomendacao_operacional": registro["recomendacao_operacional"],
        "sistema_afetado": p.get("sistema_afetado", ""), "diagnostico_provavel": p.get("diagnostico_provavel", ""),
        "falha_recorrente": p.get("falha_recorrente", ""), "situacao_preventiva": p.get("situacao_preventiva", ""),
        "pecas_provaveis": json.dumps(p.get("pecas_provaveis", []), ensure_ascii=False),
        "equipamento_substituto": p.get("equipamento_substituto", ""),
        "fontes_tecnicas": "; ".join(p.get("fontes_tecnicas", [])),
        "justificativa": registro["justificativa"],
        "revisao_humana_obrigatoria": registro["revisao_humana_obrigatoria"],
        "sobrescritas_guardrail": " | ".join(registro["sobrescritas"]),
        "modelo": registro.get("modelo", ""),
    }
    pd.DataFrame(FILA.values()).to_csv(ARQUIVO_FILA, index=False, encoding="utf-8")


def mostrar_parecer(registro: dict[str, Any]) -> None:
    p = registro["parecer"]
    if p is None:
        print(f"{registro['os_id']}: rejeitada — {registro['justificativa']}")
        return
    for rotulo, valor in [
        ("Status", p["status_triagem"]), ("Equipamento", p["equipamento_id"]), ("Sistema", p["sistema_afetado"]),
        ("Diagnóstico", f"{p['diagnostico_provavel']} (confiança {p['confianca']})"),
        ("Falha recorrente", p["falha_recorrente"]), ("Preventiva", p["situacao_preventiva"]),
        ("Prioridade", p["prioridade"]), ("Recomendação", p["recomendacao_operacional"]),
        ("Substituto", p.get("equipamento_substituto") or "—"), ("Justificativa", p["justificativa"]),
        ("Fontes", "; ".join(p["fontes_tecnicas"]) or "—")]:
        print(f"{rotulo:<17} {valor}")
    for titulo, itens in [("Procedimento", p["procedimento_recomendado"]),
                          ("Peças", [f"{x['codigo_peca']} {x['descricao']} — {x['disponibilidade']}"
                                     for x in p["pecas_provaveis"]]),
                          ("Pendências", p["pendencias"]), ("Perguntas", p["perguntas_ao_solicitante"])]:
        if itens:
            print(f"{titulo}:")
            for item in itens:
                print(f"  - {item}")
    if registro["sobrescritas"]:
        print("Guardrail:", " | ".join(registro["sobrescritas"]))
    if registro["problemas_validacao"]:
        print("Problemas de validação:", " | ".join(registro["problemas_validacao"]))


print("Lógica de negócio da Entrega 1 carregada.")

## 8.3 O State

O estado é o contrato de dados entre os nodes. `messages` usa o reducer `add_messages`, então cada node devolve só as mensagens novas. Os demais campos guardam o que foi efetivamente observado, que é a base da validação: referências recuperadas, ferramentas que responderam com sucesso e peças com estoque consultado.

In [ ]:
class TriagemState(TypedDict, total=False):
    # Entrada
    os_id: str
    percepcao: dict[str, Any]

    # Histórico LangChain, com o reducer add_messages
    messages: Annotated[list[BaseMessage], add_messages]

    # Evidências observadas
    references: list[str]
    tools_used: list[str]
    tools_with_evidence: list[str]
    pecas_consultadas: list[str]
    trajetoria: list[dict[str, Any]]

    # Contadores e falha
    model_calls: int
    tool_calls: int
    falha: str | None

    # Saída das etapas finais
    parecer: dict[str, Any]
    finalizacao: str
    validation_problems: list[str]
    sobrescritas: list[str]
    registro: dict[str, Any]


print("TriagemState definido.")

## 8.4 Nodes

O `model` substitui o centro do antigo `_decidir()`: chama o modelo com as ferramentas vinculadas e devolve a `AIMessage`. Se a chamada falhar mesmo depois das novas tentativas, o node registra a falha no estado e o grafo segue para a finalização, que gera o parecer de contingência.

O `tools` usa o `ToolNode` e registra no estado as referências, as ferramentas e as peças observadas. Se o pedido ultrapassar o limite de chamadas de ferramenta, o excedente não é executado e volta ao modelo como erro.

In [ ]:
def _resumir_resultado(nome: str, r: dict[str, Any]) -> str:
    """Uma linha sobre o resultado de uma ferramenta, para o rastro da execução."""
    if r.get("status") != "success":
        return f"{r.get('status')}: {str(r.get('mensagem', ''))[:110]}"
    if nome == "buscar_documentacao_tecnica":
        return "success: " + " | ".join(e["referencia"] for e in r["evidencias"])
    if nome == "consultar_equipamento":
        return "success: " + (r["equipamento"]["equipamento_id"] if "equipamento" in r else f"{r['quantidade']} candidato(s)")
    if nome == "consultar_historico_manutencao":
        return (f"success: {r['total_ocorrencias']} ocorrência(s) · por sistema em 90 dias: "
                f"{r['corretivas_por_sistema_ultimos_90_dias']}")
    if nome == "consultar_plano_preventiva":
        return f"success: vencidas {r['revisoes_vencidas']} · próximas {r['revisoes_proximas']}"
    if nome == "consultar_estoque_pecas":
        return "success: " + " | ".join(f"{p['codigo_peca']} {p['situacao']}" for p in r["pecas"])
    if nome == "consultar_frota_disponivel":
        return f"success: {[e['equipamento_id'] for e in r['equipamentos']] or 'nenhum disponível'}"
    return "success"


def _payload_de_tool_message(mensagem: ToolMessage) -> dict[str, Any]:
    if isinstance(mensagem.artifact, dict):
        return mensagem.artifact
    try:
        valor = json.loads(mensagem.content)
        return valor if isinstance(valor, dict) else {"data": valor}
    except Exception:
        return {"content": mensagem.content}


def node_perceive(state: TriagemState) -> dict[str, Any]:
    percepcao = perceber(ORDENS.loc[state["os_id"]])
    print("=" * 78)
    print(f"TRIAGEM {percepcao.os_id}")
    print("=" * 78)
    if not percepcao.valida:
        print(f"[PERCEIVE] OS rejeitada, sem chamar o modelo: {percepcao.motivo_rejeicao}")
        return {"percepcao": asdict(percepcao)}
    eq = percepcao.equipamento["equipamento_id"] if percepcao.equipamento else "NÃO resolvido"
    print(f"[PERCEIVE] Relato: {percepcao.contexto['relato']}")
    print(f"           Equipamento: {eq} · alarmes: {percepcao.codigos_alarme or '—'} "
          f"· gatilhos: {percepcao.gatilhos or '—'}")
    return {
        "percepcao": asdict(percepcao),
        "messages": [HumanMessage(content="Ordem de serviço para triagem:\n"
                                          + json.dumps(percepcao.contexto, ensure_ascii=False, indent=2))],
        "references": [], "tools_used": [], "tools_with_evidence": [], "pecas_consultadas": [],
        "trajetoria": [], "model_calls": 0, "tool_calls": 0, "falha": None,
    }


async def node_model(state: TriagemState) -> dict[str, Any]:
    numero = state.get("model_calls", 0) + 1
    print(f"\n[MODEL] chamada {numero}")
    mensagens = [SystemMessage(content=INSTRUCOES_SISTEMA), *state["messages"]]
    try:
        resposta: AIMessage = await llm.invocar(
            lambda chat: chat.bind_tools(tools_langchain, tool_choice="auto"),
            mensagens, inicio_da_conversa=(numero == 1))
    except Exception as erro:
        motivo = f"{type(erro).__name__}: {str(erro)[:300]}"
        print(f"[CONTINGÊNCIA] {motivo}")
        return {"falha": motivo, "model_calls": numero}
    if resposta.tool_calls:
        print(f"        o modelo pediu {len(resposta.tool_calls)} ferramenta(s)")
    else:
        print("        nenhuma ferramenta nova pedida")
    return {"messages": [resposta], "model_calls": numero}


async def node_tools(state: TriagemState) -> dict[str, Any]:
    ultima: AIMessage = state["messages"][-1]
    restantes = max(MAX_CHAMADAS_FERRAMENTA - state.get("tool_calls", 0), 0)
    permitidas, recusadas = ultima.tool_calls[:restantes], ultima.tool_calls[restantes:]

    mensagens_tool: list[ToolMessage] = []
    if permitidas:
        saida = await tool_node_base.ainvoke({"messages": [ultima.model_copy(update={"tool_calls": permitidas})]})
        mensagens_tool = list(saida["messages"])
    for chamada in recusadas:
        mensagens_tool.append(ToolMessage(
            content=json.dumps({"status": "error", "mensagem": "Limite de chamadas de ferramenta atingido."}),
            tool_call_id=chamada["id"], name=chamada["name"]))

    argumentos_por_id = {c["id"]: c["args"] for c in ultima.tool_calls}
    referencias = set(state.get("references", []))
    tools_used = set(state.get("tools_used", []))
    tools_ok = set(state.get("tools_with_evidence", []))
    pecas = set(state.get("pecas_consultadas", []))
    trajetoria = list(state.get("trajetoria", []))

    print("[TOOLS]")
    for mensagem in mensagens_tool:
        nome = mensagem.name
        payload = _payload_de_tool_message(mensagem)
        status = payload.get("status", "error")
        tools_used.add(nome)
        if status == "success":
            tools_ok.add(nome)
            if nome == "buscar_documentacao_tecnica":
                referencias.update(e["referencia"] for e in payload["evidencias"])
            if nome == "consultar_estoque_pecas":
                pecas.update(p["codigo_peca"] for p in payload["pecas"])
        argumentos = argumentos_por_id.get(mensagem.tool_call_id, {})
        trajetoria.append({"rodada": state.get("model_calls", 0), "ferramenta": nome,
                           "argumentos": argumentos, "status": status})
        print(f"   → {nome}({', '.join(f'{k}={v!r}' for k, v in argumentos.items())})")
        print(f"     {_resumir_resultado(nome, payload)}")

    return {
        "messages": mensagens_tool,
        "references": sorted(referencias), "tools_used": sorted(tools_used),
        "tools_with_evidence": sorted(tools_ok), "pecas_consultadas": sorted(pecas),
        "trajetoria": trajetoria, "tool_calls": state.get("tool_calls", 0) + len(permitidas),
    }


async def node_finalize(state: TriagemState) -> dict[str, Any]:
    print("\n[FINALIZE] parecer estruturado")
    percepcao, referencias = state["percepcao"], set(state.get("references", []))
    if state.get("falha"):
        return {"parecer": parecer_incompleto(percepcao, referencias, state["falha"]), "finalizacao": "contingencia"}

    historico = list(state["messages"])
    forcada = isinstance(historico[-1], AIMessage) and bool(historico[-1].tool_calls)
    if forcada:
        historico = historico[:-1]          # pedido de ferramentas não atendido por limite
        print("           limite atingido: o parecer usa as evidências já coletadas")
    mensagens = [SystemMessage(content=INSTRUCOES_SISTEMA), *historico, HumanMessage(content=PROMPT_FINAL)]
    try:
        parecer = await llm.invocar(
            lambda chat: chat.with_structured_output(PARECER_SCHEMA, method="json_schema"),
            mensagens, inicio_da_conversa=False)
        if not isinstance(parecer, dict):
            raise RuntimeError("Structured Output não retornou um dicionário.")
    except Exception as erro:
        motivo = f"{type(erro).__name__}: {str(erro)[:300]}"
        print(f"[CONTINGÊNCIA] {motivo}")
        return {"parecer": parecer_incompleto(percepcao, referencias, motivo), "finalizacao": "contingencia"}
    return {"parecer": parecer, "finalizacao": "forcada_por_limite" if forcada else "natural"}


def node_validate(state: TriagemState) -> dict[str, Any]:
    print("\n[VALIDATE]")
    percepcao, parecer = state["percepcao"], dict(state["parecer"])
    problemas = [] if state.get("finalizacao") == "contingencia" else validar_parecer(
        parecer, percepcao, set(state.get("references", [])),
        set(state.get("tools_with_evidence", [])), set(state.get("pecas_consultadas", [])))
    sobrescritas = aplicar_guardrail(parecer, percepcao.get("gatilhos", []))
    print("           parecer aprovado pela validação" if not problemas
          else f"           {len(problemas)} problema(s): " + " | ".join(problemas))
    for s in sobrescritas:
        print(f"[GUARDRAIL] {s}")
    return {"parecer": parecer, "validation_problems": problemas, "sobrescritas": sobrescritas}


def node_action(state: TriagemState) -> dict[str, Any]:
    percepcao = state["percepcao"]
    if not percepcao["valida"]:
        registro = {"os_id": percepcao["os_id"], "orquestracao": "Parte 1 · LangGraph",
                    "status_triagem": "rejeitada", "equipamento_id": "", "prioridade": "",
                    "recomendacao_operacional": "", "justificativa": percepcao["motivo_rejeicao"],
                    "revisao_humana_obrigatoria": True, "sobrescritas": [], "problemas_validacao": [],
                    "parecer": None, "modelo": "—", "rodadas": 0, "chamadas_ferramenta": 0,
                    "ferramentas_usadas": [], "finalizacao": "rejeitada_na_percepcao"}
    else:
        parecer, problemas = state["parecer"], state.get("validation_problems", [])
        registro = {
            "os_id": percepcao["os_id"], "orquestracao": "Parte 1 · LangGraph",
            "status_triagem": parecer["status_triagem"], "equipamento_id": parecer["equipamento_id"],
            "prioridade": parecer["prioridade"], "recomendacao_operacional": parecer["recomendacao_operacional"],
            "justificativa": parecer["justificativa"],
            "revisao_humana_obrigatoria": bool(problemas) or parecer["status_triagem"] == "incompleta",
            "problemas_validacao": problemas, "sobrescritas": state.get("sobrescritas", []),
            "gatilhos": percepcao.get("gatilhos", []), "trajetoria": state.get("trajetoria", []),
            "referencias_recuperadas": state.get("references", []), "parecer": parecer,
            "modelo": llm.modelo, "rodadas": state.get("model_calls", 0) + (state.get("finalizacao") != "contingencia"),
            "chamadas_ferramenta": state.get("tool_calls", 0), "ferramentas_usadas": state.get("tools_used", []),
            "finalizacao": state.get("finalizacao", ""),
        }
    agir(registro)
    print(f"\n[ACTION] {registro['status_triagem']} · prioridade {registro['prioridade'] or '—'} · "
          f"{registro['recomendacao_operacional'] or '—'} · finalização: {registro['finalizacao']}"
          + (" · REVISÃO HUMANA OBRIGATÓRIA" if registro["revisao_humana_obrigatoria"] else ""))
    return {"registro": registro}


print("Nodes definidos.")

## 8.5 Roteamento

Os `if` que ficavam dentro do laço da Entrega 1 viram funções de roteamento, usadas em conditional edges:

- depois de `perceive`: OS inválida vai direto para `action`, sem chamar o modelo;
- depois de `model`: com ferramentas pedidas e dentro dos limites, vai para `tools`; caso contrário, para `finalize`.

O limite de rodadas da Entrega 1 continua: no máximo 6 chamadas ao modelo por triagem, contando a finalização.

In [ ]:
def route_after_perceive(state: TriagemState) -> str:
    destino = "model" if state["percepcao"]["valida"] else "action"
    print(f"[ROUTING] PERCEIVE → {destino.upper()}")
    return destino


def route_after_model(state: TriagemState) -> str:
    ultima = state["messages"][-1]
    pediu_ferramentas = isinstance(ultima, AIMessage) and bool(ultima.tool_calls)
    dentro_dos_limites = (state.get("model_calls", 0) < MAX_RODADAS - 1
                          and state.get("tool_calls", 0) < MAX_CHAMADAS_FERRAMENTA)
    if state.get("falha"):
        destino = "finalize"
    elif pediu_ferramentas and dentro_dos_limites:
        destino = "tools"
    else:
        destino = "finalize"
        if pediu_ferramentas:
            print("[LIMITE] limite de rodadas ou de ferramentas atingido")
    print(f"[ROUTING] MODEL → {destino.upper()}")
    return destino

## 8.6 Montando o `StateGraph`

A topologia inteira fica explícita em poucas linhas. O desenho abaixo é gerado a partir do grafo compilado.

In [ ]:
builder = StateGraph(TriagemState)

builder.add_node("perceive", node_perceive)
builder.add_node("model", node_model)
builder.add_node("tools", node_tools)
builder.add_node("finalize", node_finalize)
builder.add_node("validate", node_validate)
builder.add_node("action", node_action)

builder.add_edge(START, "perceive")
builder.add_conditional_edges("perceive", route_after_perceive, {"model": "model", "action": "action"})
builder.add_conditional_edges("model", route_after_model, {"tools": "tools", "finalize": "finalize"})
builder.add_edge("tools", "model")
builder.add_edge("finalize", "validate")
builder.add_edge("validate", "action")
builder.add_edge("action", END)

grafo_parte1 = builder.compile()
print("StateGraph compilado.")

try:
    display(Image(data=grafo_parte1.get_graph().draw_mermaid_png()))
except Exception as erro:
    print(f"Não foi possível renderizar o PNG ({type(erro).__name__}). Representação Mermaid:\n")
    print(grafo_parte1.get_graph().draw_mermaid())

## 8.7 Execução: OS-2026-0101

Mesma OS da demonstração da Entrega 1: alarme TL-214 na escavadeira EH-07, que já teve duas corretivas hidráulicas nos últimos 45 dias. O rastro mostra cada node e cada transição.

In [ ]:
estado_parte1 = await grafo_parte1.ainvoke({"os_id": "OS-2026-0101"}, config={"recursion_limit": 25})
registro_parte1 = estado_parte1["registro"]
print("\n" + llm.resumo())

In [ ]:
mostrar_parecer(registro_parte1)

## 8.8 Entrega 1 × Parte 1

| Responsabilidade | Entrega 1 | Parte 1 |
|---|---|---|
| Estado | `EstadoTriagem` mantido pelo runtime | `TriagemState` compartilhado pelos nodes |
| Histórico | Lista de `types.Content` montada à mão | `messages` com o reducer `add_messages` |
| Ferramentas | Registro `nome → função` | Servidor MCP, adaptador para `StructuredTool` e `ToolNode` |
| Chamada do modelo | Dentro do laço | Node `model` |
| Decisão de rota | `if` no laço | Conditional edges |
| Ciclo | `while` | Edge `tools → model` |
| Finalização | Ferramenta terminal `emitir_parecer` | Node `finalize` com Structured Output |
| Validação e guardrail | Funções no fim do método | Node `validate` |
| Fila | `agir()` | Node `action` |

A lógica de negócio é a mesma. Mudou a forma como a execução é orquestrada.

In [ ]:
await cliente_mcp.encerrar()
print("Sessão MCP da Parte 1 encerrada.")

---
# 9. Parte 2 — Os agentes especialistas

Cada especialista tem duas camadas:

```text
┌─────────────────────────────────────────┐
│        CAMADA DE PROTOCOLO (A2A)        │
│ Agent Card · Message · Task · Artifact  │
├─────────────────────────────────────────┤
│        LÓGICA INTERNA DO AGENTE         │
│ Gemini · Vector RAG · MCP · regras      │
└─────────────────────────────────────────┘
```

O A2A não define como o agente pensa. Ele padroniza como outro agente descobre o especialista, envia uma subtarefa, acompanha a Task e recebe o Artifact.

## 9.1 O servidor A2A

| Agente | Lógica interna |
|---|---|
| **Knowledge** | Três buscas fixas no Vector RAG (sintoma ou alarme na ficha técnica; prioridade e recorrência; critérios de bloqueio) e uma síntese estruturada pelo Gemini. As referências e os códigos de peça da síntese são filtrados contra as evidências recuperadas |
| **Operations** | Cliente MCP com o servidor no perfil `operacional`. Consulta sempre o cadastro e, se o equipamento existir, o histórico, a preventiva, o estoque das peças da ficha técnica e a frota disponível. Não usa o modelo |
| **Strategy** | Uma chamada ao Gemini com o schema do parecer da Entrega 1. Não tem ferramentas. As fontes técnicas do parecer são filtradas contra as referências recebidas do Knowledge |

As chamadas ao Gemini dentro do servidor usam a mesma resiliência da seção 4.5: novas tentativas e modelos de reserva. Se um especialista falhar mesmo assim, a Task termina como `FAILED` e o Supervisor decide o que fazer.

O script cria três `AgentCard`s, três `AgentSkill`s, três `AgentExecutor`s e três endpoints JSON-RPC, com o Agent Card de cada um em `/.well-known/agent-card.json`.

In [ ]:
SERVIDOR_A2A = Path("servidor_a2a_apollo.py")

CODIGO_SERVIDOR_A2A = r'''"""Servidor A2A da Construtora Apollo — agentes especialistas da triagem de manutenção.

Um único processo HTTP hospeda três agentes A2A logicamente independentes:
- Knowledge Agent: interpreta a documentação técnica (Vector RAG);
- Operations Agent: levanta os fatos operacionais (cliente MCP, perfil operacional);
- Strategy Agent: emite o parecer a partir dos Artifacts recebidos (sem ferramentas).

Cada agente publica o próprio Agent Card e o próprio endpoint JSON-RPC. Um único processo
facilita a execução no Colab; a comunicação continua usando o protocolo A2A.
"""
from __future__ import annotations

import asyncio
import json
import os
import re
import sys
from functools import lru_cache
from pathlib import Path
from typing import Any, Awaitable, Callable

import chromadb
import httpx
import uvicorn
from google import genai
from google.genai import types
from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client
from starlette.applications import Starlette

from a2a.helpers import get_message_text, new_task_from_user_message, new_text_message, new_text_part
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill
from a2a.types.a2a_pb2 import TaskState
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH

HOST = os.environ.get("APOLLO_A2A_HOST", "127.0.0.1")
PORT = int(os.environ.get("APOLLO_A2A_PORT", "8125"))
PUBLIC_URL = os.environ.get("APOLLO_A2A_PUBLIC_URL", f"http://{HOST}:{PORT}")

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
MODELOS = os.environ["APOLLO_MODELOS"].split(",")
DIRETORIO_CHROMA = os.environ["APOLLO_CHROMA_DIR"]
NOME_COLECAO = os.environ["APOLLO_CHROMA_COLECAO"]
MCP_SERVER_SCRIPT = os.environ["APOLLO_MCP_SERVER"]
PASTA_DADOS = os.environ["APOLLO_DADOS"]
DATA_REFERENCIA = os.environ["APOLLO_DATA_REFERENCIA"]

MODELO_EMBEDDING = "gemini-embedding-001"
DIMENSAO_EMBEDDING = 768
TENTATIVAS = 3
ESPERA_BASE = 30.0
ESPERA_MAXIMA_ACEITAVEL = 90

gemini = genai.Client(api_key=GEMINI_API_KEY,
                      http_options=types.HttpOptions(timeout=180_000,
                                                     retry_options=types.HttpRetryOptions(attempts=1)))
knowledge_collection = chromadb.PersistentClient(path=DIRETORIO_CHROMA).get_collection(NOME_COLECAO)


# ---------------------------------------------------------------------
# Helpers compartilhados
# ---------------------------------------------------------------------

_indice_modelo = 0


def _retry_delay(erro: Exception) -> float | None:
    texto = str(erro) + json.dumps(getattr(erro, "details", None) or {}, default=str)
    achado = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", texto)
    return float(achado.group(1)) if achado else None


async def gerar_estruturado(prompt: str, schema: dict[str, Any]) -> tuple[dict[str, Any], str]:
    """Chamada ao Gemini com saída estruturada, retry e modelos de reserva (resiliência da Entrega 1).

    Cada chamada de especialista é uma conversa nova, então a troca de modelo é segura a qualquer momento.
    """
    global _indice_modelo
    while True:
        modelo = MODELOS[_indice_modelo]
        ultimo_erro: Exception | None = None
        for tentativa in range(1, TENTATIVAS + 1):
            try:
                resposta = await gemini.aio.models.generate_content(
                    model=modelo, contents=prompt,
                    config=types.GenerateContentConfig(response_mime_type="application/json",
                                                       response_json_schema=schema))
                if not isinstance(resposta.parsed, dict):
                    raise RuntimeError("o modelo não devolveu saída estruturada")
                return resposta.parsed, modelo
            except Exception as erro:
                ultimo_erro = erro
                codigo = getattr(erro, "code", None)
                transitorio = (isinstance(erro, (httpx.TimeoutException, httpx.TransportError))
                               or codigo in {429, 500, 502, 503, 504})
                if not transitorio:
                    raise
                sugerido = _retry_delay(erro)
                if (codigo == 429 and sugerido and sugerido > ESPERA_MAXIMA_ACEITAVEL) or tentativa == TENTATIVAS:
                    break
                espera = sugerido or ESPERA_BASE * 2 ** (tentativa - 1)
                print(f"[{modelo}] {codigo or type(erro).__name__} — nova tentativa em {espera:.0f}s", flush=True)
                await asyncio.sleep(espera)
        if _indice_modelo + 1 >= len(MODELOS):
            raise RuntimeError(f"nenhum modelo disponível; último erro: {ultimo_erro}")
        _indice_modelo += 1
        print(f"[{modelo}] indisponível; próximas chamadas usam {MODELOS[_indice_modelo]}", flush=True)


@lru_cache(maxsize=256)
def embedding_query(texto: str) -> tuple[float, ...]:
    resposta = gemini.models.embed_content(
        model=MODELO_EMBEDDING, contents=[texto],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=DIMENSAO_EMBEDDING))
    return tuple(resposta.embeddings[0].values)


def retrieve_knowledge(query: str, ficha_tecnica: str | None, top_k: int = 3) -> list[dict[str, Any]]:
    filtro = {"aplicacao": {"$in": [ficha_tecnica, "geral"]}} if ficha_tecnica else None
    resposta = knowledge_collection.query(query_embeddings=[list(embedding_query(query))], n_results=top_k,
                                          where=filtro, include=["documents", "metadatas", "distances"])
    return [{"referencia": meta["referencia"], "documento": meta["titulo_documento"], "secao": meta["titulo"],
             "trecho": texto, "distance": round(float(distancia), 4)}
            for texto, meta, distancia in zip(resposta["documents"][0], resposta["metadatas"][0],
                                              resposta["distances"][0])]


def mcp_payload(result: Any) -> dict[str, Any]:
    payload = getattr(result, "structured_content", None)
    if isinstance(payload, dict):
        return payload
    bruto = "\n".join(getattr(b, "text", "") or "" for b in getattr(result, "content", []) or []).strip()
    return {"status": "error", "mensagem": bruto or "resultado sem conteúdo estruturado"}


# ---------------------------------------------------------------------
# Knowledge Agent
# ---------------------------------------------------------------------

KNOWLEDGE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "resumo_tecnico": {"type": "string"},
        "orientacoes_diagnostico": {"type": "array", "items": {"type": "string"}},
        "regras_prioridade_recorrencia": {"type": "array", "items": {"type": "string"}},
        "criterios_seguranca": {"type": "array", "items": {"type": "string"}},
        "pecas_associadas": {"type": "array", "items": {"type": "string"}},
        "referencias": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["resumo_tecnico", "orientacoes_diagnostico", "regras_prioridade_recorrencia",
                 "criterios_seguranca", "pecas_associadas", "referencias"],
}


async def knowledge_brain(payload: dict[str, Any]) -> dict[str, Any]:
    percepcao = payload["percepcao"]
    objective = payload.get("objective", "Interpretar a documentação técnica aplicável à OS.")
    equipamento = percepcao.get("equipamento") or {}
    ficha = equipamento.get("ficha_tecnica")
    gatilhos = percepcao.get("gatilhos", [])
    relato = percepcao["contexto"]["relato"]

    # Três buscas fixas, uma por assunto
    consultas = [
        (f"{relato} {' '.join(percepcao.get('codigos_alarme', []))}", ficha),
        ("matriz de prioridade de atendimento e regra de falha recorrente da política de manutenção", None),
        (f"situações de bloqueio obrigatório do equipamento: {', '.join(gatilhos) if gatilhos else relato[:150]}", None),
    ]
    evidencias: dict[str, dict[str, Any]] = {}
    for consulta, filtro in consultas:
        for evidencia in retrieve_knowledge(consulta, filtro):
            evidencias.setdefault(evidencia["referencia"], evidencia)
    lista = list(evidencias.values())
    referencias_validas = set(evidencias)
    contexto = "\n\n".join(f"[{e['referencia']}] {e['secao']}\n{e['trecho']}" for e in lista)

    prompt = f"""Você é o Knowledge Agent da Construtora Apollo S.A.

Sua única responsabilidade é interpretar a documentação técnica de manutenção aplicável à ordem de serviço.
Você não consulta cadastro, histórico, preventiva, estoque nem frota.

OBJETIVO DELEGADO:
{objective}

ORDEM DE SERVIÇO:
{json.dumps(percepcao["contexto"], ensure_ascii=False, indent=2)}

EVIDÊNCIAS RECUPERADAS DA DOCUMENTAÇÃO:
{contexto}

Produza uma síntese objetiva: orientações de diagnóstico, regras de prioridade e recorrência, critérios de
segurança e os códigos APL das peças associadas ao problema. Cite SOMENTE referências presentes nas evidências
acima, no formato exato (ex.: "FTM-CAT320 § 5.2"). Não invente fatos operacionais.
"""
    analise, modelo = await gerar_estruturado(prompt, KNOWLEDGE_SCHEMA)
    codigos_nas_evidencias = set(re.findall(r"APL-\d{5}", contexto))
    analise["referencias"] = [r for r in analise.get("referencias", []) if r in referencias_validas]
    analise["pecas_associadas"] = [p for p in analise.get("pecas_associadas", []) if p in codigos_nas_evidencias]

    return {"status": "success", "agent": "Knowledge Agent", "modelo": modelo, "analise": analise,
            "evidencias": lista}


# ---------------------------------------------------------------------
# Operations Agent
# ---------------------------------------------------------------------

async def operations_brain(payload: dict[str, Any]) -> dict[str, Any]:
    objective = payload.get("objective", "Levantar os fatos operacionais do equipamento.")
    informado = payload.get("equipamento_informado", "")
    obra_id = payload.get("obra_id")

    env = {"APOLLO_MCP_PERFIL": "operacional", "APOLLO_DADOS": PASTA_DADOS,
           "APOLLO_DATA_REFERENCIA": DATA_REFERENCIA}
    params = StdioServerParameters(command=sys.executable, args=[MCP_SERVER_SCRIPT], env=env, cwd=str(Path.cwd()))

    resultados: dict[str, Any] = {}
    with open("operations_mcp_stderr.log", "a", encoding="utf-8", buffering=1) as errlog:
        async with Client(stdio_client(params, errlog=errlog)) as client:

            async def chamar(nome: str, argumentos: dict[str, Any]) -> dict[str, Any]:
                resultado = await client.call_tool(nome, argumentos)
                resultados[nome] = mcp_payload(resultado)
                return resultados[nome]

            cadastro = await chamar("consultar_equipamento", {"equipamento_id": informado, "obra_id": obra_id}
                                    if informado else {"obra_id": obra_id})
            if cadastro.get("status") == "success" and "equipamento" in cadastro:
                eq = cadastro["equipamento"]
                await chamar("consultar_historico_manutencao", {"equipamento_id": eq["equipamento_id"]})
                await chamar("consultar_plano_preventiva", {"equipamento_id": eq["equipamento_id"]})
                await chamar("consultar_estoque_pecas", {"ficha_tecnica": eq["ficha_tecnica"], "obra_id": eq["obra_id"]})
                await chamar("consultar_frota_disponivel", {"tipo": eq["tipo"], "obra_id": eq["obra_id"]})
                observacao = "Equipamento identificado: consultas operacionais completas."
            else:
                observacao = ("Equipamento não identificado no cadastro: histórico, preventiva, estoque e frota "
                              "não foram consultados.")

    return {"status": "success", "agent": "Operations Agent", "objective": objective,
            "fatos_operacionais": resultados, "tools_executadas": list(resultados), "observacao": observacao}


# ---------------------------------------------------------------------
# Strategy Agent
# ---------------------------------------------------------------------

SISTEMAS = ["Hidráulico", "Motor", "Arrefecimento", "Elétrico", "Trem de força", "Freios",
            "Direção", "Suspensão", "Material rodante", "Estrutura e implementos"]

PARECER_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "status_triagem": {"type": "string", "enum": ["concluida", "devolvida"]},
        "equipamento_id": {"type": "string"},
        "sistema_afetado": {"type": "string", "enum": SISTEMAS + ["Indeterminado"]},
        "diagnostico_provavel": {"type": "string"},
        "confianca": {"type": "string", "enum": ["alta", "media", "baixa"]},
        "procedimento_recomendado": {"type": "array", "maxItems": 8, "items": {"type": "string"}},
        "pecas_provaveis": {"type": "array", "maxItems": 8, "items": {
            "type": "object", "additionalProperties": False,
            "properties": {"codigo_peca": {"type": "string", "pattern": "^APL-[0-9]{5}$"},
                           "descricao": {"type": "string"},
                           "disponibilidade": {"type": "string", "enum": ["na_obra", "em_outro_almoxarifado",
                                                                          "sem_estoque", "nao_verificada"]}},
            "required": ["codigo_peca", "descricao", "disponibilidade"]}},
        "falha_recorrente": {"type": "string", "enum": ["sim", "nao", "nao_verificado"]},
        "situacao_preventiva": {"type": "string", "enum": ["em_dia", "proxima", "vencida", "nao_verificada"]},
        "prioridade": {"type": "string", "enum": ["P1", "P2", "P3", "P4"]},
        "recomendacao_operacional": {"type": "string",
                                     "enum": ["liberar", "liberar_com_restricao", "bloquear_ate_inspecao"]},
        "equipamento_substituto": {"type": "string"},
        "justificativa": {"type": "string"},
        "fontes_tecnicas": {"type": "array", "items": {"type": "string"}},
        "pendencias": {"type": "array", "items": {"type": "string"}},
        "perguntas_ao_solicitante": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["status_triagem", "equipamento_id", "sistema_afetado", "diagnostico_provavel", "confianca",
                 "procedimento_recomendado", "pecas_provaveis", "falha_recorrente", "situacao_preventiva",
                 "prioridade", "recomendacao_operacional", "justificativa", "fontes_tecnicas",
                 "pendencias", "perguntas_ao_solicitante"],
}


async def strategy_brain(payload: dict[str, Any]) -> dict[str, Any]:
    percepcao = payload["percepcao"]
    objective = payload.get("objective", "Emitir o parecer de triagem a partir das evidências.")
    specialist_results = payload.get("specialist_results", {})
    if not isinstance(specialist_results, dict):
        raise TypeError("specialist_results deve ser um dict.")

    prompt = f"""Você é o Strategy Agent da Construtora Apollo S.A.

Sua responsabilidade é emitir o parecer de triagem da ordem de serviço para o planejador de manutenção,
usando somente os Artifacts recebidos. Você RECOMENDA; não bloqueia nem libera equipamentos.

Você NÃO possui ferramentas. Você NÃO consulta documentação nem sistemas. Não invente informação ausente.

OBJETIVO DELEGADO:
{objective}

ORDEM DE SERVIÇO:
{json.dumps(percepcao["contexto"], ensure_ascii=False, indent=2)}

ARTIFACTS DISPONÍVEIS:
{json.dumps(specialist_results, ensure_ascii=False, indent=2, default=str)}

REGRAS DO PARECER
- Fatos operacionais só se constarem no Artifact do Operations Agent: sem histórico, falha_recorrente =
  nao_verificado; sem preventiva, situacao_preventiva = nao_verificada; sem estoque, disponibilidade = nao_verificada.
- fontes_tecnicas: apenas referências do Artifact do Knowledge Agent, no formato exato.
- Equipamento não identificado ou relato insuficiente: status_triagem = devolvida, com perguntas_ao_solicitante,
  e equipamento_id = NAO_IDENTIFICADO quando não houver identificação.
- Se a frota foi consultada e há substituto disponível em qualquer obra, informe-o em equipamento_substituto.
- Não liste em pecas_provaveis componente já trocado em corretiva recente sem diagnóstico que justifique nova troca.
- Gatilho de segurança na OS: recomende bloquear_ate_inspecao.
- Registre em pendencias o que não foi possível verificar.
"""
    decisao, modelo = await gerar_estruturado(prompt, PARECER_SCHEMA)

    # Grounding: fontes técnicas só podem vir do Artifact do Knowledge Agent
    referencias_validas = set(specialist_results.get("knowledge", {}).get("analise", {}).get("referencias", []))
    descartadas = [r for r in decisao.get("fontes_tecnicas", []) if r not in referencias_validas]
    decisao["fontes_tecnicas"] = [r for r in decisao.get("fontes_tecnicas", []) if r in referencias_validas]

    return {"status": "success", "agent": "Strategy Agent", "modelo": modelo, "objective": objective,
            "decision": decisao, "referencias_descartadas": descartadas}


# ---------------------------------------------------------------------
# Adaptador do protocolo A2A
# ---------------------------------------------------------------------

Brain = Callable[[dict[str, Any]], Awaitable[dict[str, Any]]]


class JsonBrainExecutor(AgentExecutor):
    """Adapta a lógica interna de um agente ao ciclo de vida de uma Task A2A."""

    def __init__(self, agent_name: str, brain: Brain) -> None:
        self.agent_name = agent_name
        self.brain = brain

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        if context.current_task:
            task = context.current_task
        else:
            task = new_task_from_user_message(context.message)
            await event_queue.enqueue_event(task)

        updater = TaskUpdater(event_queue=event_queue, task_id=task.id, context_id=task.context_id)
        await updater.update_status(state=TaskState.TASK_STATE_WORKING,
                                    message=new_text_message(f"{self.agent_name}: processando a subtarefa."))
        try:
            payload = json.loads(get_message_text(context.message))
            resultado = await self.brain(payload)
            await updater.add_artifact(parts=[new_text_part(
                text=json.dumps(resultado, ensure_ascii=False, default=str), media_type="application/json")])
            await updater.update_status(state=TaskState.TASK_STATE_COMPLETED,
                                        message=new_text_message(f"{self.agent_name}: subtarefa concluída."))
        except Exception as erro:
            await updater.update_status(
                state=TaskState.TASK_STATE_FAILED,
                message=new_text_message(f"{self.agent_name}: falha na execução — {type(erro).__name__}: {erro}"))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Cancelamento não implementado.")


def build_card(path: str, name: str, description: str, skill_id: str, skill_name: str,
               skill_description: str, examples: list[str]) -> AgentCard:
    return AgentCard(
        name=name,
        description=description,
        version="1.0.0",
        default_input_modes=["text/plain"],
        default_output_modes=["application/json"],
        capabilities=AgentCapabilities(streaming=True),
        supported_interfaces=[AgentInterface(protocol_binding="JSONRPC", url=f"{PUBLIC_URL}{path}",
                                             protocol_version="1.0")],
        skills=[AgentSkill(id=skill_id, name=skill_name, description=skill_description,
                           input_modes=["text/plain"], output_modes=["application/json"],
                           tags=["manutencao", "apollo", "a2a"], examples=examples)],
    )


AGENTS = [
    ("/knowledge",
     build_card("/knowledge", "Apollo Knowledge Agent",
                "Especialista na documentação técnica de manutenção: fichas técnicas, catálogo de alarmes, "
                "procedimento de segurança e política de manutenção.",
                "interpretar_documentacao_tecnica", "Interpretar documentação técnica",
                "Recupera evidências via Vector RAG e interpreta diagnóstico, prioridade, recorrência, "
                "segurança e peças associadas.",
                ["O que a documentação diz sobre o alarme TL-214 na escavadeira CAT 320?"]),
     JsonBrainExecutor("Knowledge Agent", knowledge_brain)),
    ("/operations",
     build_card("/operations", "Apollo Operations Agent",
                "Especialista em fatos operacionais: cadastro da frota, histórico de corretivas, preventiva, "
                "estoque de peças e equipamentos disponíveis.",
                "levantar_fatos_operacionais", "Levantar fatos operacionais",
                "Consulta cadastro, histórico, preventiva, estoque e frota usando MCP.",
                ["Levante histórico, preventiva, estoque e substitutos do equipamento EH-07."]),
     JsonBrainExecutor("Operations Agent", operations_brain)),
    ("/strategy",
     build_card("/strategy", "Apollo Strategy Agent",
                "Especialista em emitir o parecer de triagem de manutenção a partir dos Artifacts recebidos.",
                "emitir_parecer_triagem", "Emitir parecer de triagem",
                "Recebe as evidências dos especialistas e produz o parecer estruturado da triagem.",
                ["Dadas estas regras e estes fatos, qual parecer emitir para a OS?"]),
     JsonBrainExecutor("Strategy Agent", strategy_brain)),
]


def build_app() -> Starlette:
    routes = []
    for path, card, executor in AGENTS:
        handler = DefaultRequestHandler(agent_executor=executor, task_store=InMemoryTaskStore(), agent_card=card)
        routes.extend(create_agent_card_routes(card, card_url=f"{path}{AGENT_CARD_WELL_KNOWN_PATH}"))
        routes.extend(create_jsonrpc_routes(handler, rpc_url=path))
    return Starlette(routes=routes)


if __name__ == "__main__":
    print(f"Apollo A2A host em {PUBLIC_URL}")
    for path, card, _ in AGENTS:
        print(f"  {PUBLIC_URL}{path} -> {card.name}")
    uvicorn.run(build_app(), host=HOST, port=PORT)
'''

SERVIDOR_A2A.write_text(CODIGO_SERVIDOR_A2A, encoding="utf-8")
print(f"Servidor A2A gravado em {SERVIDOR_A2A.resolve()} ({len(CODIGO_SERVIDOR_A2A.splitlines())} linhas)")

## 9.2 Iniciando o servidor A2A

Os três agentes ficam expostos em `/knowledge`, `/operations` e `/strategy`. Fisicamente é um único processo `uvicorn`; logicamente são três agentes independentes, cada um com o próprio endpoint.

O processo é iniciado com `subprocess.Popen()`, porque `uvicorn.run()` bloqueia a execução: o servidor precisa continuar ativo enquanto o notebook segue com o Supervisor.

In [ ]:
A2A_HOST = "127.0.0.1"
A2A_PORT = 8125
A2A_BASE_URL = f"http://{A2A_HOST}:{A2A_PORT}"
AGENT_URLS = {
    "knowledge": f"{A2A_BASE_URL}/knowledge",
    "operations": f"{A2A_BASE_URL}/operations",
    "strategy": f"{A2A_BASE_URL}/strategy",
}
LOG_A2A = Path("a2a_server.log")

# Se a célula for executada de novo, encerra o processo anterior
if "processo_a2a" in globals():
    try:
        if processo_a2a.poll() is None:
            processo_a2a.terminate()
            processo_a2a.wait(timeout=5)
    except Exception:
        pass

env_a2a = os.environ.copy()
env_a2a.update({
    "PYTHONUNBUFFERED": "1",
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "APOLLO_MODELOS": ",".join(MODELOS_DISPONIVEIS),
    "APOLLO_A2A_HOST": A2A_HOST,
    "APOLLO_A2A_PORT": str(A2A_PORT),
    "APOLLO_A2A_PUBLIC_URL": A2A_BASE_URL,
    "APOLLO_CHROMA_DIR": str(Path(DIRETORIO_CHROMA).resolve()),
    "APOLLO_CHROMA_COLECAO": NOME_COLECAO,
    "APOLLO_MCP_SERVER": str(SERVIDOR_MCP.resolve()),
    "APOLLO_DADOS": str(PASTA_DADOS),
    "APOLLO_DATA_REFERENCIA": DATA_REFERENCIA.isoformat(),
})

log_handle = open(LOG_A2A, "w", encoding="utf-8", buffering=1)
processo_a2a = subprocess.Popen([sys.executable, str(SERVIDOR_A2A.resolve())], env=env_a2a,
                                stdout=log_handle, stderr=subprocess.STDOUT, cwd=str(Path.cwd()))
print("Processo A2A iniciado. PID:", processo_a2a.pid)

## 9.3 Verificação de prontidão

O processo HTTP leva alguns segundos para subir. Antes de seguir, o notebook tenta obter os três Agent Cards até que todos respondam. Nenhuma chamada ao modelo é feita aqui.

In [ ]:
async def esperar_agent_cards(urls: dict[str, str], tentativas: int = 30, espera_s: float = 1.0) -> None:
    async with httpx.AsyncClient() as http_client:
        pendentes = dict(urls)
        for _ in range(tentativas):
            concluidos = []
            for nome, base_url in pendentes.items():
                try:
                    await A2ACardResolver(http_client, base_url).get_agent_card()
                    concluidos.append(nome)
                except Exception:
                    pass
            for nome in concluidos:
                pendentes.pop(nome, None)
            if not pendentes:
                return
            await asyncio.sleep(espera_s)
    log_handle.flush()
    trecho_log = LOG_A2A.read_text(encoding="utf-8")[-4000:] if LOG_A2A.exists() else ""
    raise RuntimeError(f"Servidor A2A não ficou pronto. Pendentes: {list(pendentes)}\n\nLog:\n{trecho_log}")


await esperar_agent_cards(AGENT_URLS)
print("Servidor A2A pronto.")
for nome, url in AGENT_URLS.items():
    print(f"  {nome:<11} {url}")

## 9.4 Descoberta dos agentes

Para cada URL conhecida, o `A2ACardResolver` busca o Agent Card publicado pelo agente. É a partir desses cards que o Supervisor vai saber quais especialistas existem, o que cada um faz e como acioná-los.

In [ ]:
async def descobrir_cards(urls: dict[str, str]) -> dict[str, Any]:
    cards: dict[str, Any] = {}
    async with httpx.AsyncClient() as http_client:
        for chave, base_url in urls.items():
            cards[chave] = await A2ACardResolver(http_client, base_url).get_agent_card()
    return cards


agent_cards = await descobrir_cards(AGENT_URLS)

for chave, card in agent_cards.items():
    print("\n" + "=" * 78)
    print(chave.upper())
    print("=" * 78)
    print("name:", card.name)
    print("description:", card.description)
    print("version:", card.version)
    for interface in card.supported_interfaces:
        print("interface:", interface.protocol_binding, interface.protocol_version, interface.url)
    for skill in card.skills:
        print(f"skill: {skill.id} — {skill.description}")

Até aqui o notebook só descobriu os contratos dos agentes: identidade, skills e interfaces. Nenhuma subtarefa foi delegada ainda. Isso começa na próxima seção, com o Supervisor.